# Análise de vendas de jogos


> **Briefing:**
>
> Você trabalha para a loja online Ice, que vende videogames no mundo todo. As avaliações de usuários e especialistas, gêneros, plataformas (por exemplo, Xbox ou PlayStation) e dados históricos sobre vendas de jogos estão disponíveis em fontes abertas.
> Você precisa identificar padrões que determinam se um jogo tem sucesso ou não. Isso vai permitir que você identifique possíveis sucessos e planeje campanhas publicitárias.
>
> Os dados disponibilizados remontam a 2016. Vamos imaginar que estamos em dezembro de 2016 e você está planejando uma campanha para 2017.
>
> O conjunto de dados contém uma coluna de "rating" (classificação) que armazena a classificação ESRB de cada jogo. O Entertainment Software Rating Board avalia o conteúdo de um jogo e atribui uma classificação etária, como Teen (Adolescente) ou Mature (Adulto).

**Descrição dos dados:**

-  Name (nome)
-  Platform (plataforma)
-  Year_of_Release (Ano de lançamento)
-  Genre (gênero)
-  NA_sales (vendas norte-americanas em milhões de USD)
-  EU_sales (vendas na Europa em milhões de USD)
-  JP_sales (vendas no Japão em milhões de USD)
-  Other_sales (vendas em outros países em milhões de USD)
-  Critic_Score (Pontuação crítica) (máximo de 100)
-  User_Score (Pontuação dos usuários) (máximo de 10)
-  Classificação (ESRB)

## Plano de trabalho

1. Pré-processamento
    - <input type="checkbox" checked> Renomear colunas    
    - <input type="checkbox" checked> Avaliar colunas com dados vazios
        - <input type="checkbox" checked> Preencher, remover ou manter colunas com dados vazios
    - <input type="checkbox" checked> Investigar duplicatas
        - <input type="checkbox" checked> Corrigir ou remover duplicatas
    - <input type="checkbox" checked> Corrigir tipos de dados
    - <input type="checkbox" checked> Enriquecimento dos dados
        - <input type="checkbox" checked> Criar coluna total de vendas, com a soma de todas as regiões
        - <input type="checkbox" checked> Converter as avaliações de críticos e usuários para a mesma escala, para poder compará-las
2. Análise exploratória 
    - <input type="checkbox" checked> Quantos jogos foram lançados por ano?
    - <input type="checkbox" checked> Analisar as vendas por plataforma:
        - <input type="checkbox" checked> Quais são as plataformas mais populares?
        - <input type="checkbox" checked> Quantos jogos foram lançados para cada uma por ano?
        - <input type="checkbox" checked> Por quanto tempo uma plataforma se mantém relevante?
    - <input type="checkbox" checked> Definir quais dados são mais relevantes para uma análise preditiva para 2017
    - <input type="checkbox" checked> Analisar quais são as plataformas com maior potencial de vendas
        - <input type="checkbox" checked> Comparar a distribuição da distribuição das vendas entre as plataformas
        - <input type="checkbox" checked> Como as avaliações afetam as vendas?
    - <input type="checkbox" checked> Como se distribuem os jogos por gênero?
        - <input type="checkbox" checked> Quais são os gêneros mais populares? E os menos?
        - <input type="checkbox" checked> O que podemos generalizar sobre os gêneros com vendas altas e baixas?
3. Categorização por região
    - <input type="checkbox" checked> Quais são as principais plataformas de cada região?
        - <input type="checkbox" checked> Quanto elas detem de mercado em cada região?
    - <input type="checkbox" checked> Quais são os gêneros mais populares em cada região?
    - <input type="checkbox" checked> Como a classificação ESRB afeta as vendas em cada região?
4. Testes de hipóteses
    - <input type="checkbox"> Testar hipótese: As classificações médias dos usuários das plataformas Xbox One e PC são as mesmas.
    - <input type="checkbox"> Testar hipótese: As classificações médias de usuários para os gêneros Action (ação) e Sports (esportes) são diferentes.
5. Conclusão

## Inicialização

In [704]:
# Importando bibiliotecas
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px

from scipy import stats

In [463]:
# Importando o dataset
try:
    df = pd.read_csv('datasets/games.csv') # caminho local no windows
except FileNotFoundError:
    df = pd.read_csv('/datasets/games.csv') # caminho no ambiente da tripleten


In [464]:
# Visualizando os dados
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16715 entries, 0 to 16714
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16713 non-null  object 
 1   Platform         16715 non-null  object 
 2   Year_of_Release  16446 non-null  float64
 3   Genre            16713 non-null  object 
 4   NA_sales         16715 non-null  float64
 5   EU_sales         16715 non-null  float64
 6   JP_sales         16715 non-null  float64
 7   Other_sales      16715 non-null  float64
 8   Critic_Score     8137 non-null   float64
 9   User_Score       10014 non-null  object 
 10  Rating           9949 non-null   object 
dtypes: float64(6), object(5)
memory usage: 1.4+ MB


In [465]:
# Visualizando as primeiras linhas do dataset
df.head()

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [466]:
# Visualizando algumas amostras dos dados
df.sample(10, random_state=42)

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
3485,London 2012: The Official Video Game of the Ol...,X360,2012.0,Sports,0.07,0.44,0.00,0.07,NaN,NaN,NaN
5500,Etrian Odyssey,DS,2007.0,Role-Playing,0.19,0.02,0.09,0.02,75.0,8.5,T
16713,Spirits & Spells,GBA,2003.0,Platform,0.01,0.00,0.00,0.00,NaN,NaN,NaN
9489,RollerCoaster Tycoon,XB,2003.0,Strategy,0.10,0.03,0.00,0.00,62.0,8.3,E
12993,Rhapsody: A Musical Adventure,DS,2008.0,Role-Playing,0.05,0.00,0.00,0.00,67.0,6.8,E
12749,Men in Black The Series: Crashdown,PS,2001.0,Shooter,0.03,0.02,0.00,0.00,NaN,NaN,NaN
7270,Samurai Jack: The Amulet of Time,GBA,2003.0,Platform,0.16,0.06,0.00,0.00,63.0,tbd,T
2935,King Kong,2600,1981.0,Action,0.65,0.04,0.00,0.01,NaN,NaN,NaN
8739,Pressure Cooker,2600,1982.0,Action,0.14,0.01,0.00,0.00,NaN,NaN,NaN
2227,Cars,GC,2006.0,Racing,0.72,0.19,0.00,0.03,71.0,7,E


In [467]:
# Resumo estatístico das colunas numéricas
df.describe()

,Year_of_Release,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score
count,16446.000000,16715.000000,16715.000000,16715.000000,16715.000000,8137.000000
mean,2006.484616,0.263377,0.145060,0.077617,0.047342,68.967679
std,5.877050,0.813604,0.503339,0.308853,0.186731,13.938165
min,1980.000000,0.000000,0.000000,0.000000,0.000000,13.000000
25%,2003.000000,0.000000,0.000000,0.000000,0.000000,60.000000
50%,2007.000000,0.080000,0.020000,0.000000,0.010000,71.000000
75%,2010.000000,0.240000,0.110000,0.040000,0.030000,79.000000
max,2016.000000,41.360000,28.960000,10.220000,10.570000,98.000000


### Observações

**Formatação:**

- Renomear colunas para `snake_case` 

**Dados vazios:**

- Temos dados vazios em 6 colunas:
    - `Name`, `Year_of_Release`, `Genre` tem poucos valores vazios, avaliar possibilidade de remover as linhas
    - `Critics_Score`, `User_Score`, `Rating`: tem muitos valores, pode ser relevante pensar uma estratégia de preenchimento

**Tipos de dados:**

- Corrigir `Year_of_Release` de float para int, já que anos são inteiros
- `User_Score` precisa ser float, para realizarmos operações matemáticas, mas existem algumas linhas com a string 'tbd' que precisamos corrigir.

## Pré-processamento

In [468]:
# Renomeando colunas para snake_case
df.columns = df.columns.str.lower().str.replace(' ', '_')

# checando as colunas renomeadas
print(df.columns)
df.info()

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16715 entries, 0 to 16714
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16713 non-null  object 
 1   platform         16715 non-null  object 
 2   year_of_release  16446 non-null  float64
 3   genre            16713 non-null  object 
 4   na_sales         16715 non-null  float64
 5   eu_sales         16715 non-null  float64
 6   jp_sales         16715 non-null  float64
 7   other_sales      16715 non-null  float64
 8   critic_score     8137 non-null   float64
 9   user_score       10014 non-null  object 
 10  rating           9949 non-null   object 
dtypes: float64(6), object(5)
memory usage: 1.4+ MB


### Lidando com dados vazios

#### Colunas com poucas linhas vazias

In [469]:
# Checando vazios da coluna name
df[df['name'].isnull()]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
659,NaN,GEN,1993.0,NaN,1.78,0.53,0.00,0.08,NaN,NaN,NaN
14244,NaN,GEN,1993.0,NaN,0.00,0.00,0.03,0.00,NaN,NaN,NaN


In [470]:
# Checando vazios da coluna genre
df[df['genre'].isna()]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
659,NaN,GEN,1993.0,NaN,1.78,0.53,0.00,0.08,NaN,NaN,NaN
14244,NaN,GEN,1993.0,NaN,0.00,0.00,0.03,0.00,NaN,NaN,NaN


In [471]:
# Checando vazios da coluna year_of_release
df_year_null = df[df['year_of_release'].isna()]
print('Número de valores vazios na coluna year_of_release:', df['year_of_release'].isna().sum())
print(f'Porcentagem de jogos sem ano de lançamento: {(len(df_year_null) / len(df)) * 100:.2f}%')
print()
print(df_year_null['platform'].value_counts())

Número de valores vazios na coluna year_of_release: 269
Porcentagem de jogos sem ano de lançamento: 1.61%

platform
PS2     34
Wii     34
X360    30
DS      30
PS3     25
XB      21
2600    17
PC      17
PSP     16
GC      14
GBA     11
3DS      8
PS       7
N64      3
GB       1
PSV      1
Name: count, dtype: int64


In [472]:
df_year_null.head(10)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
183,Madden NFL 2004,PS2,NaN,Sports,4.26,0.26,0.01,0.71,94.0,8.5,E
377,FIFA Soccer 2004,PS2,NaN,Sports,0.59,2.36,0.04,0.51,84.0,6.4,E
456,LEGO Batman: The Videogame,Wii,NaN,Action,1.80,0.97,0.00,0.29,74.0,7.9,E10+
475,wwe Smackdown vs. Raw 2006,PS2,NaN,Fighting,1.57,1.02,0.00,0.41,NaN,NaN,NaN
609,Space Invaders,2600,NaN,Shooter,2.36,0.14,0.00,0.03,NaN,NaN,NaN
627,Rock Band,X360,NaN,Misc,1.93,0.33,0.00,0.21,92.0,8.2,T
657,Frogger's Adventures: Temple of the Frog,GBA,NaN,Adventure,2.15,0.18,0.00,0.07,73.0,tbd,E
678,LEGO Indiana Jones: The Original Adventures,Wii,NaN,Action,1.51,0.61,0.00,0.21,78.0,6.6,E10+
719,Call of Duty 3,Wii,NaN,Shooter,1.17,0.84,0.00,0.23,69.0,6.7,T
805,Rock Band,Wii,NaN,Misc,1.33,0.56,0.00,0.20,80.0,6.3,T


In [473]:
# Removendo linhas valores vazios em year_of_release, name e genre
df = df.dropna(subset=['year_of_release', 'name', 'genre'])

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16444 entries, 0 to 16714
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16444 non-null  object 
 1   platform         16444 non-null  object 
 2   year_of_release  16444 non-null  float64
 3   genre            16444 non-null  object 
 4   na_sales         16444 non-null  float64
 5   eu_sales         16444 non-null  float64
 6   jp_sales         16444 non-null  float64
 7   other_sales      16444 non-null  float64
 8   critic_score     7983 non-null   float64
 9   user_score       9839 non-null   object 
 10  rating           9768 non-null   object 
dtypes: float64(6), object(5)
memory usage: 1.5+ MB


##### Observações:
- As dados vazios de `name` e `genre` estão nas mesmas linhas, ambas foram removidas
- Decidi remover as linhas com dados vazio em `year_of_release` pois:
    - Apesar de alguns jogos terem vendas expressivas, o ano de lançamento é uma variável central para a análise temporal, preenche-lo pode tornar os resultados menos confiáveis
    - **1,61%** é um percentual aceitável dos dados para desconsiderar da análise.

#### Colunas com muitas linhas vazias

In [474]:
# Checando vazios da coluna critic_score
df_critic_null = df[df['critic_score'].isna()]
print('Número de valores vazios na coluna critic_score:', df['critic_score'].isna().sum())
print(f'Porcentagem de jogos sem critic_score: {(len(df_critic_null) / len(df)) * 100:.2f}%')

# Checando os valores únicos da coluna critic_score, incluindo os NaN
df['critic_score'].value_counts(dropna=False)

Número de valores vazios na coluna critic_score: 8461
Porcentagem de jogos sem critic_score: 51.45%


critic_score
NaN     8461
70.0     252
71.0     248
75.0     240
80.0     235
        ... 
20.0       3
29.0       3
21.0       1
17.0       1
13.0       1
Name: count, Length: 82, dtype: int64

In [475]:
# Checando vazios da coluna user_score
df_user_null = df[df['user_score'].isna()]
print('Número de valores vazios na coluna user_score:', df['user_score'].isna().sum())
print(f'Porcentagem de jogos sem user_score: {(len(df_user_null) / len(df)) * 100:.2f}%')

# Checando os valores únicos da coluna user_score, incluindo os NaN
df['user_score'].value_counts(dropna=False)

Número de valores vazios na coluna user_score: 6605
Porcentagem de jogos sem user_score: 40.17%


user_score
NaN    6605
tbd    2376
7.8     322
8       285
8.2     276
       ... 
1.9       2
9.6       2
0.9       2
0         1
9.7       1
Name: count, Length: 97, dtype: int64

In [476]:
# Checando vazios da coluna rating
df_rating_null = df[df['rating'].isna()]
print('Número de valores vazios na coluna rating:', df['rating'].isna().sum())
print(f'Porcentagem de jogos sem rating: {(len(df_rating_null) / len(df)) * 100:.2f}%')

# Checando os valores únicos da coluna rating, incluindo os NaN
df['rating'].value_counts(dropna=False)

Número de valores vazios na coluna rating: 6676
Porcentagem de jogos sem rating: 40.60%


rating
NaN     6676
E       3921
T       2905
M       1536
E10+    1393
EC         8
K-A        3
AO         1
RP         1
Name: count, dtype: int64

In [477]:
# Corrigindo os valores 'tbd' para NaN na coluna user_score
df['user_score'] = df['user_score'].replace('tbd', np.nan)

# Checando vazios da coluna user_score novamente após a correção
df_user_null = df[df['user_score'].isna()]
print('Número de valores vazios na coluna user_score:', df['user_score'].isna().sum())
print(f'Porcentagem de jogos sem user_score: {(len(df_user_null) / len(df)) * 100:.2f}%')

# Checando por virgulas nos scores e substituindo por pontos para garantir que sejam interpretados como números decimais
print('\nNúmero de valores com vírgula na coluna user_score: ', df['user_score'].str.contains(',', na=False).sum())


Número de valores vazios na coluna user_score: 8981
Porcentagem de jogos sem user_score: 54.62%

Número de valores com vírgula na coluna user_score:  0


In [478]:
# Substituindo NaN por 'N/A' na coluna rating
df['rating'] = df['rating'].fillna('N/A')

# Checando os valores únicos da coluna rating novamente, incluindo os NaN
print(df['rating'].value_counts(dropna=False))

rating
N/A     6676
E       3921
T       2905
M       1536
E10+    1393
EC         8
K-A        3
AO         1
RP         1
Name: count, dtype: int64


In [479]:
# Checando o dataframe após as limpezas
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16444 entries, 0 to 16714
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16444 non-null  object 
 1   platform         16444 non-null  object 
 2   year_of_release  16444 non-null  float64
 3   genre            16444 non-null  object 
 4   na_sales         16444 non-null  float64
 5   eu_sales         16444 non-null  float64
 6   jp_sales         16444 non-null  float64
 7   other_sales      16444 non-null  float64
 8   critic_score     7983 non-null   float64
 9   user_score       7463 non-null   object 
 10  rating           16444 non-null  object 
dtypes: float64(6), object(5)
memory usage: 1.5+ MB


#### Observações:
- Para coluna `rating` decidi preencher os valores vazios com N/A (Not Available) indicando que a classificação não está disponível. 
- Para `user_score` e `critic_score` a decisão foi de manter os valores vazios, pois:
    - Removê-los, significa perder um volume significativo dos dados (51%, apenas no caso de `critic_score`)
    - Substituir por 0 ou pela média/mediana introduziria um viés considerável nos dados e poderia distorcer análises
    - Mantê-los signfica que ao fazer análises de avaliação, trabalharemos com um subset menor dos dados, mas não compremetemos os resultados.
- Para `user_score`, ainda tinhamos uma quantidade considerável de linhas preenchidas com 'tbd' (To Be Defined), que decidi tratar como valores vazios. Substituindo-os para `NaN` para poder realizar a conversão para float.

### Checando duplicatas

In [480]:
# Checando por linhas duplicadas
print('Quantidade de linhas duplicadas:', df.duplicated().sum())

# Checando por linhas duplicadas considerando name, platform, year_of_release e genre
print('Quantidade de linhas duplicadas considerando name, platform, year_of_release e genre:', df.duplicated(subset=['name', 'platform', 'year_of_release', 'genre']).sum())

# Checando por duplicadas considerando apenas as colunas name, platform e year_of_release
print('Quantidade de linhas duplicadas considerando name, platform e year_of_release:', df.duplicated(subset=['name', 'platform', 'year_of_release']).sum())

# Checando por linhas duplicadas considerando name, platform, year_of_release e genre
print('Quantidade de linhas duplicadas considerando name, platform:', df.duplicated(subset=['name', 'platform']).sum())


Quantidade de linhas duplicadas: 0
Quantidade de linhas duplicadas considerando name, platform, year_of_release e genre: 1
Quantidade de linhas duplicadas considerando name, platform e year_of_release: 1
Quantidade de linhas duplicadas considerando name, platform: 3


In [481]:
# Imprimindo as linhas duplicadas considerando name e platform
duplicated_rows = df[df.duplicated(subset=['name', 'platform', 'year_of_release', 'genre'], keep=False)]
display(duplicated_rows)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
604,Madden NFL 13,PS3,2012.0,Sports,2.11,0.22,0.0,0.23,83.0,5.5,E
16230,Madden NFL 13,PS3,2012.0,Sports,0.00,0.01,0.0,0.00,83.0,5.5,E


In [482]:
# Removendo a linha duplicada
df = df.drop(index=16230)

# Checando o resultado
duplicated_rows = df[df.duplicated(subset=['name', 'platform', 'year_of_release', 'genre'], keep=False)]
display(duplicated_rows)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating


##### Observações:

- Encontrei uma instância de registro duplicado para o mesmo jogo, plataforma e ano de lançamento. 
- A coluna duplicada foi tratada como erro de registro, já que reproduzia o mesmos valores nos metadados e nas avaliações, além de 0 vendas em todas as regiões exceto em `eu_sales` com 0.01 (10 mil dólares), consideravelmente menor que o desvio padrão para `eu_sales` (~0.5), portanto estatisticamente pouco relevante. 

### Corrigindo tipos de dados

In [483]:
# Convertendo year_of_release para inteiro
df['year_of_release'] = df['year_of_release'].astype(int)

# Convertendo user_score para float
df['user_score'] = pd.to_numeric(df['user_score'],errors='coerce')

# Checando se houve algum problema na conversão
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16443 entries, 0 to 16714
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16443 non-null  object 
 1   platform         16443 non-null  object 
 2   year_of_release  16443 non-null  int64  
 3   genre            16443 non-null  object 
 4   na_sales         16443 non-null  float64
 5   eu_sales         16443 non-null  float64
 6   jp_sales         16443 non-null  float64
 7   other_sales      16443 non-null  float64
 8   critic_score     7982 non-null   float64
 9   user_score       7462 non-null   float64
 10  rating           16443 non-null  object 
dtypes: float64(6), int64(1), object(4)
memory usage: 1.5+ MB


### Enriquecendo os dados

In [484]:
# Criando a coluna total_sales
df['total_sales'] = df['na_sales'] + df['eu_sales'] + df['jp_sales'] + df['other_sales']

# Checando o dataframe após a adição da coluna
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16443 entries, 0 to 16714
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16443 non-null  object 
 1   platform         16443 non-null  object 
 2   year_of_release  16443 non-null  int64  
 3   genre            16443 non-null  object 
 4   na_sales         16443 non-null  float64
 5   eu_sales         16443 non-null  float64
 6   jp_sales         16443 non-null  float64
 7   other_sales      16443 non-null  float64
 8   critic_score     7982 non-null   float64
 9   user_score       7462 non-null   float64
 10  rating           16443 non-null  object 
 11  total_sales      16443 non-null  float64
dtypes: float64(7), int64(1), object(4)
memory usage: 1.6+ MB


In [485]:
# Exibindo uma amostra dos dados para verificar a nova coluna total_sales
df.sample(5, random_state=42)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,total_sales
4797,NCAA GameBreaker 2000,PS,1999,Sports,0.22,0.15,0.00,0.03,NaN,NaN,N/A,0.40
15039,Steel Battalion: Line of Contact,XB,2004,Simulation,0.02,0.01,0.00,0.00,70.0,8.0,T,0.03
9479,Shrek Super Party,XB,2002,Misc,0.10,0.03,0.00,0.00,33.0,8.0,E,0.13
8281,Hello Kitty: Happy Party Pals,GBA,2005,Misc,0.12,0.05,0.00,0.00,NaN,NaN,N/A,0.17
323,Call of Duty: Ghosts,PS4,2013,Shooter,1.78,1.43,0.05,0.57,78.0,3.7,M,3.83


In [486]:
# Multiplicando a coluna user_score por 10 para que fique na mesma escala do critic_score
df['user_score'] = df['user_score'] * 10

# Exibindo uma amostra dos dados para verificar a nova escala da coluna user_score
df.sample(5, random_state=42)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,total_sales
4797,NCAA GameBreaker 2000,PS,1999,Sports,0.22,0.15,0.00,0.03,NaN,NaN,N/A,0.40
15039,Steel Battalion: Line of Contact,XB,2004,Simulation,0.02,0.01,0.00,0.00,70.0,80.0,T,0.03
9479,Shrek Super Party,XB,2002,Misc,0.10,0.03,0.00,0.00,33.0,80.0,E,0.13
8281,Hello Kitty: Happy Party Pals,GBA,2005,Misc,0.12,0.05,0.00,0.00,NaN,NaN,N/A,0.17
323,Call of Duty: Ghosts,PS4,2013,Shooter,1.78,1.43,0.05,0.57,78.0,37.0,M,3.83


##### Observações:

- Foi criada uma coluna `total_sales` para poder fazer analises do total de vendas, independente da região.
- Multiplique a coluna `user_score` por 10, para que fique na mesma escala de `critic_score`, facilitando a comparação entre as duas em visualizações e eventuais operações artiméticas. 

## Análise exploratória

### Análise por ano

In [487]:
# Agrupando jogos por ano de lançamento
df_grouped_year = df.groupby('year_of_release').agg({
    'name': 'count',
    'total_sales': 'sum',
    'critic_score': 'mean',
    'user_score': 'mean',
    'platform': 'nunique'
}).reset_index()

df_grouped_year.rename(columns={'name': 'game_count', 'platform': 'platform_count'}, inplace=True)

df_grouped_year.sample(10, random_state=42)

,year_of_release,game_count,total_sales,critic_score,user_score,platform_count
17,1997,289,201.07,85.294118,84.722222,6
13,1993,60,45.99,NaN,NaN,5
4,1984,14,50.35,NaN,NaN,2
29,2009,1426,658.82,67.554531,69.907166,7
35,2015,606,267.98,72.871111,66.659933,10
25,2005,939,457.82,68.819847,75.096055,8
6,1986,21,37.08,NaN,NaN,2
26,2006,1006,517.71,67.338710,72.472826,10
24,2004,762,418.68,69.393939,77.505112,7
16,1996,263,199.15,89.875000,84.000000,8


In [488]:
# Criando um gráfico de com o total de jogos lançados por ano e a evolução das vendas totais ao longo dos anos
fig = px.bar(df_grouped_year, x='year_of_release', y='game_count', title='Número de Jogos Lançados por Ano')
fig.add_scatter(x=df_grouped_year['year_of_release'], y=df_grouped_year['total_sales'], mode='lines+markers', name='Vendas Totais')
fig.update_layout(xaxis_title='Ano de Lançamento', yaxis_title='Número de Jogos / Vendas Totais (em milhões USD)')
fig.show() 

##### Observações:

- Os dados mais relantes para nosso objetivo, **planejar uma campanha para 2017** estão na fatia dos anos 2010, pois representam o período mais recente do mercado.
- Existe um boom de lançamentos em torno de 2008 (entre 2005 e 2011).
- Podemos desconsiderar os dados dos anos 80 e primeira metade dos anos 90, por terem um baixo volume e serem temporalmente bastante distantes do nosso objetivo. 
- A redução após 2012 não necessariamente representa uma retração da indústria de jogos, mas pode refletir limitações do dataset, mudanças no modelo de distribuição digital, transição de gerações de consoles e a consolidação de jogos como serviço, reduzindo a frequência de novos lançamentos físicos. *Vamos continuar a investigação, analisando as plataformas representadas nos dados*

### Análise das plataformas

In [489]:
# Identificando as plataformas presentes no dataset
print('Plataformas distintas no dataset:', df['platform'].unique())
print('Número de plataformas distintas:', df['platform'].nunique())

Plataformas distintas no dataset: ['Wii' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XOne' 'WiiU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']
Número de plataformas distintas: 31


In [490]:
# Agregando dados por plataforma
df_grouped_platform = df.groupby('platform').agg({
    'name': 'count',
    'total_sales': 'sum',
    'year_of_release': 'nunique'
}).sort_values(by='total_sales', ascending=False).reset_index()

df_grouped_platform.rename(columns={'name': 'game_count', 'year_of_release': 'year_count'}, inplace=True)

display(df_grouped_platform.head(20))

top_10_platforms = df_grouped_platform['platform'].head(10).tolist()

print('Top 10 plataformas mais populares por total em vendas:', top_10_platforms)

,platform,game_count,total_sales,year_count
0,PS2,2127,1233.56,12
1,X360,1232,961.24,12
2,PS3,1305,931.33,11
3,Wii,1286,891.18,11
4,DS,2121,802.78,11
5,PS,1190,727.58,10
6,PS4,392,314.14,4
7,GBA,811,312.88,8
8,PSP,1193,289.53,12
9,3DS,512,257.81,6


Top 10 plataformas mais populares por total em vendas: ['PS2', 'X360', 'PS3', 'Wii', 'DS', 'PS', 'PS4', 'GBA', 'PSP', '3DS']


In [491]:
# Criando um gráfico para visualizar a evolução das vendas totais por plataforma, usando plotly express
fig = px.bar(df_grouped_platform.sort_values('total_sales'), x='total_sales', y='platform', title='Vendas Totais por Plataforma')
fig.update_layout(yaxis_title='Plataforma', xaxis_title='Vendas Totais (em milhões USD)')
fig.show()

In [492]:
# Agrupando por ano e plataforma para analisar a evolução das vendas por plataforma ao longo dos anos
df_grouped_year_platform = df.groupby(['year_of_release', 'platform']).agg({
    'name': 'count',
    'total_sales': 'sum',
    'critic_score': 'mean',
    'user_score': 'mean'
}).reset_index()

# Renomeando colunas
df_grouped_year_platform.rename(columns={'name': 'game_count'}, inplace=True)

In [493]:
# Criando um heatmap para analisar a evolução das vendas por plataforma ao longo dos anos
fig = px.density_heatmap(df_grouped_year_platform, x='year_of_release', y='platform', z='total_sales', title='Evolução das Vendas por Plataforma ao Longo dos Anos', color_continuous_scale='Reds', width=1200, height=800)
fig.update_layout(xaxis_title='Ano de Lançamento', yaxis_title='Plataforma', coloraxis_colorbar_title='Vendas Totais (em milhões USD)')
fig.show()

In [494]:
# Criando um heatmap para analisar a evolução dos lançamentos de jogos por plataforma ao longo dos anos
fig = px.density_heatmap(df_grouped_year_platform, x='year_of_release', y='platform', z='game_count', title='Lançamentos por Plataforma ao Longo dos Anos', color_continuous_scale='Reds', width=1200, height=800)
fig.update_layout(xaxis_title='Ano de Lançamento', yaxis_title='Plataforma', coloraxis_colorbar_title='Número de Jogos')
fig.show()

#### Observações:

- O ciclo de vida de uma plataforma parece durar em **torno de 10 anos**, como pode ser observados nos mapas de calor de vendas e lançamentos de jogos:
    - Plataformas muito populares como o PlayStation (PS), PlayStation2 (PS2) e o Nintendo DS (DS) já não registram vendas a partir de 2015.
    - O período de entre 2010 a 2014 marca a transição para uma nova geração de consoles: XboxOne, PS4, WiiU, 3DS. Observável pelas células do topo direito do heatmap se tornando mais escuras em volume de vendas e/ou lançamentos, enquanto a geração anterior entra em declínio.  
- *Alguns fatores podem ajudar a explicar a queda após 2012:*
    - O dataset parece estar mais enviesado a lançamentos de consoles, tendo menos representação para PC e não registrando jogos para dispositivos mobile (Android e iOS). Após 2012, a Steam (plataforma de venda de jogos para PC da Valve) se estabelece a ganha popularidade. 
    - Outro dado relevate, é que nas gerações mais recentes, a indústria de jogos vem investindo mais ciclos de desenvolvimento mais longos, micro-transações, com pacotes de expansão e DLCs e distribuição digital, ao invés de mídia física. Todos esses fatores parecem estar ausentes do dataset, o que pode explicar a queda de lançamentos e registros de vendas. Algumas dessas tendências, podem ser observadas [relatório da ESA](https://www.theesa.com/u-s-consumer-video-game-spending-totaled-57-2-billion-in-2023).
- **Decisões para aprofundar a análise:**
    - Incluir nas análises de avaliações, classificação e genêros: 
        - As 10 plataformas mais populares em vendas: 'PS2', 'X360', 'PS3', 'Wii', 'DS', 'PS', 'PS4', 'GBA', 'PSP', '3DS'
        - Mais as plataformas novas e com lançamentos recentes (incluindo PC), que não estão nessa lista: 'PC', 'XOne', 'WiiU', 'PSV' 
        - Com  base nessas plataformas, o período analisado será de **1995 até 2016**. Com análises mais direcionadas conduzidas em cima da transição de plataformas observando o período de **2005 a 2016**.

### Filtrando os dados mais relevantes

In [495]:
## Criando uma lista com as plataformas selecionadas:

# Criando uma lista com os critérios de seleção: plataformas com jogos com vendas totais maiores que 0 e lançados a partir de 2012
current_platforms = df[(df['total_sales'] > 0) & (df['year_of_release'] >= 2010)]['platform'].unique().tolist()

print('Plataformas com jogos lançados a partir de 2010 e vendas registradas:', current_platforms)

# Incluindo as top10 plataformas mais populares por total em vendas na lista de plataformas selecionadas
selected_platforms = current_platforms.copy()
for platform in top_10_platforms:
    if platform not in current_platforms:
        selected_platforms.append(platform)

print('Plataformas selecionadas para análise:', selected_platforms)

Plataformas com jogos lançados a partir de 2010 e vendas registradas: ['X360', 'PS3', 'DS', 'PS4', '3DS', 'Wii', 'XOne', 'WiiU', 'PC', 'PSP', 'PSV', 'PS2']
Plataformas selecionadas para análise: ['X360', 'PS3', 'DS', 'PS4', '3DS', 'Wii', 'XOne', 'WiiU', 'PC', 'PSP', 'PSV', 'PS2', 'PS', 'GBA']


In [496]:
## Criando um segundo dataframe apenas com as plataformas com jogos com vendas registradas e lançados a partir de 2010
# Esse dataframe será usado para análises de padrões e tendências mais recentes no mercado
df_recent = df[
    (df['year_of_release'] >= 2010) &
    (df['platform'].isin(current_platforms)) &
    (df['total_sales'] > 0) 
].reset_index()

df_recent.info()
df_recent.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5276 entries, 0 to 5275
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   index            5276 non-null   int64  
 1   name             5276 non-null   object 
 2   platform         5276 non-null   object 
 3   year_of_release  5276 non-null   int64  
 4   genre            5276 non-null   object 
 5   na_sales         5276 non-null   float64
 6   eu_sales         5276 non-null   float64
 7   jp_sales         5276 non-null   float64
 8   other_sales      5276 non-null   float64
 9   critic_score     2311 non-null   float64
 10  user_score       2498 non-null   float64
 11  rating           5276 non-null   object 
 12  total_sales      5276 non-null   float64
dtypes: float64(7), int64(2), object(4)
memory usage: 536.0+ KB


,index,year_of_release,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,total_sales
count,5276.000000,5276.000000,5276.000000,5276.000000,5276.000000,5276.000000,2311.000000,2498.000000,5276.000000
mean,9036.269333,2012.357468,0.214255,0.159439,0.057835,0.050697,70.448723,66.787830,0.482227
std,4950.131811,2.034574,0.613145,0.464229,0.227132,0.150650,13.562006,15.213361,1.236082
min,14.000000,2010.000000,0.000000,0.000000,0.000000,0.000000,13.000000,2.000000,0.010000
25%,4792.750000,2011.000000,0.000000,0.000000,0.000000,0.000000,62.000000,59.000000,0.040000
50%,9423.000000,2012.000000,0.050000,0.020000,0.000000,0.010000,73.000000,70.000000,0.130000
75%,13486.500000,2014.000000,0.172500,0.130000,0.040000,0.040000,80.000000,78.000000,0.400000
max,16714.000000,2016.000000,15.000000,9.090000,5.650000,3.960000,97.000000,93.000000,21.820000


In [497]:
## Criando um dataframe filtrado com as decisões da última etapa, ou seja, jogos lançados a partir de 1995, com vendas registradas e pertencentes às plataformas selecionadas

# Esse data frame será usado para análises de padrões em generos e scores e sua correlação com vendas ao longos dos anos. 
# Ele é mais abrangente do que o df_recent, pois inclui jogos lançados a partir de 1995, permitindo uma análise histórica mais completa

df_full = df[
    (df['year_of_release'] >= 1995) &
    (df['platform'].isin(selected_platforms)) &
    (df['total_sales'] > 0) 
].reset_index()

df_full.info()
df_full.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13916 entries, 0 to 13915
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   index            13916 non-null  int64  
 1   name             13916 non-null  object 
 2   platform         13916 non-null  object 
 3   year_of_release  13916 non-null  int64  
 4   genre            13916 non-null  object 
 5   na_sales         13916 non-null  float64
 6   eu_sales         13916 non-null  float64
 7   jp_sales         13916 non-null  float64
 8   other_sales      13916 non-null  float64
 9   critic_score     6821 non-null   float64
 10  user_score       6516 non-null   float64
 11  rating           13916 non-null  object 
 12  total_sales      13916 non-null  float64
dtypes: float64(7), int64(2), object(4)
memory usage: 1.4+ MB


,index,year_of_release,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,total_sales
count,13916.000000,13916.000000,13916.000000,13916.000000,13916.000000,13916.000000,6821.000000,6516.000000,13916.000000
mean,8488.211555,2007.815895,0.249491,0.154516,0.063525,0.053529,68.820554,70.627379,0.521061
std,4872.916146,4.750614,0.731331,0.530882,0.263946,0.202260,13.944340,14.973660,1.506706
min,0.000000,1995.000000,0.000000,0.000000,0.000000,0.000000,13.000000,0.000000,0.010000
25%,4238.750000,2005.000000,0.000000,0.000000,0.000000,0.000000,60.000000,63.000000,0.060000
50%,8524.500000,2008.000000,0.070000,0.020000,0.000000,0.010000,71.000000,74.000000,0.160000
75%,12771.250000,2011.000000,0.230000,0.120000,0.030000,0.040000,79.000000,81.000000,0.460000
max,16714.000000,2016.000000,41.360000,28.960000,6.500000,10.570000,98.000000,97.000000,82.540000


### Definindo as plataformas com potencial de lucro


In [498]:
# Agregando por plataforma e ano de lançamento para analisar a evolução das vendas por plataforma ao longo dos anos
df_recent_by_platform = df_recent.groupby(['year_of_release', 'platform']).agg({
    'name': 'count',
    'total_sales': 'sum',
    'critic_score': 'mean',
    'user_score': 'mean'
}).reset_index()

df_recent_by_platform.rename(columns={'name': 'game_count', 'critic_score': 'mean_critic_score', 'user_score': 'mean_user_score'}, inplace=True)
df_recent_by_platform.head(10)

,year_of_release,platform,game_count,total_sales,mean_critic_score,mean_user_score
0,2010,DS,323,85.02,67.732143,72.628571
1,2010,PC,90,24.28,73.016393,68.223881
2,2010,PS2,38,5.64,77.000000,71.333333
3,2010,PS3,181,142.17,67.507937,67.229508
4,2010,PSP,188,35.04,68.697674,72.386364
5,2010,Wii,253,127.95,65.194805,71.241935
6,2010,X360,182,170.03,65.713235,66.893130
7,2011,3DS,116,63.20,62.693548,64.894737
8,2011,DS,153,26.18,64.684211,69.800000
9,2011,PC,139,35.03,72.703297,64.990099


In [499]:
# Criando um boxplot para analisar a distribuição das vendas por plataforma ao longo dos anos
fig = px.box(
    df_recent,
    x='platform',
    y='total_sales',
    points='outliers',
    hover_name='name',
    title='Distribuição das Vendas por Plataforma (Jogos Lançados a partir de 2010)',
    width=1400,
    height=800,
)
fig.update_traces(marker=dict(size=6), hovertemplate='<b>%{hovertext}</b><br>Plataforma: %{x}<br>Vendas: %{y:.2f}M')
fig.update_layout(xaxis_title='Plataforma', yaxis_title='Vendas Totais (em milhões USD)')
fig.show()

Parece que o mercado de jogos é largamente influenciado por grandes títulos. Para uma visualização geral por plataforma, vamos olhar para o total de vendas agrupado por ano de lançamento dos jogos. 

In [500]:
# Plotando novamente com os dados agrupados por ano, para suavizar os outliters
fig = px.box(
    df_recent_by_platform,
    x='platform',
    y='total_sales',
    points='outliers',
    title='Distribuição das Vendas por Plataforma (Dados Agrupados por Ano)',
    width=1400,
    height=800,
)
fig.update_layout(xaxis_title='Plataforma', yaxis_title='Vendas Totais (em milhões USD)')
fig.show()

In [501]:
# Criando um gráfico de linha para visualizar a evolução das vendas totais ao longo dos anos
# Filtrando para os dados até 2015, já que 2016 pode ter dados incompletos e afetar a análise de tendências
fig = px.line(df_recent_by_platform[df_recent_by_platform['year_of_release'] <= 2015], x='year_of_release', y='total_sales', color='platform', title='Evolução das Vendas Totais por Plataforma (Jogos Lançados a partir de 2010)', width=1400, height=800)
fig.update_layout(xaxis_title='Ano de Lançamento', yaxis_title='Vendas Totais (em milhões USD)')
fig.show()

In [502]:
# Criando um gráfico de barra para visualizar a evolução das vendas totais ao longo dos anos
fig = px.bar(df_recent_by_platform, 
             x='year_of_release', 
             y='total_sales', 
             color='platform', 
             title='Evolução das Vendas Totais por Plataforma (Jogos Lançados a partir de 2010)', 
             width=1400, 
             height=800,
             barmode='stack')
fig.update_layout(xaxis_title='Ano de Lançamento', yaxis_title='Vendas Totais (em milhões USD)')
fig.show()

Parece que as únicas plataformas em ascenção são as novas: Xbox One (XOne) e Playstation 4 (PS4). O novo console da Nintendo WiiU vem tendo dificuldade de se estabelecer no mercado e repetir o sucesso do Wii. Vamos observar as medianas para entender uma visão do mercado com menos impacto dos grandes títulos.

In [503]:
# Calculando a mediana de vendas por plataforma para identificar quais plataformas têm bom potencial de vendas fora dos outliers
sales_by_platform = df_recent.groupby(['platform',]).agg(
                                                        {'total_sales': ['median', 'mean', 'sum'],
                                                        }).reset_index()

sales_by_platform.columns = ['platform', 'median_sales', 'mean_sales', 'total_sales']
sales_by_platform = sales_by_platform.sort_values('median_sales', ascending=False).reset_index(drop=True)

print(sales_by_platform)

   platform  median_sales  mean_sales  total_sales
0      X360          0.27    0.809426       550.41
1       PS3          0.23    0.661858       587.73
2      WiiU          0.22    0.559116        82.19
3      XOne          0.22    0.645020       159.32
4       PS4          0.20    0.801378       314.14
5       Wii          0.18    0.495489       222.97
6       3DS          0.12    0.503535       257.81
7        DS          0.10    0.244083       123.75
8        PC          0.08    0.254614       121.96
9       PS2          0.06    0.135333         6.09
10      PSP          0.05    0.128100        64.05
11      PSV          0.05    0.125431        53.81


In [504]:
# Criando um gráfico de linha para visualizar a evolução das medianas de vendas por plataforma ao longo dos anos
fig = px.bar(sales_by_platform, x='platform', y='median_sales', color='platform', title='Evolução da Mediana de Vendas por Plataforma (Jogos Lançados a partir de 2010)', width=1400, height=800)
fig.update_layout(xaxis_title='Plataforma', yaxis_title='Mediana de Vendas (em milhões USD)')
fig.show()

In [505]:
# Criando um gráfico de barras com o volume lançamentos por plataforma
fig = px.bar(df_recent_by_platform, x='platform', y='game_count', title='Lançamentos por Plataforma')
fig.update_layout(xaxis_title='Plataforma', yaxis_title='Número de Jogos')
fig.show()

#### Descobertas:

- O total de vendas de jogos é bastante influenciado por grandes títulos: alguns títulos de plataformas muito populares podem sozinho trazer mais receita que a receita anual de algumas plataformas. Kinect Adventures! para XBox 360 conquistou mais receita que todas as vendas de jogos para Wii e WiiU em 2012.
- Podemos observar XBox One e Playstation 4 ganhando espaço no mercado, enquanto suas gerações anteriores diminuem em vendas.
- Os consoles da Sony (Playstation) e Microsofot (Xbox) detem as maiores fatias do mercado, com os consoles da Nintendo tendo maior sucesso com portáteis (DS, 3DS) e dificuldade de conquistar muitas vendas com o sucessor do Wii, o WiiU. 
    - Ainda assim, desconsiderando o ano de 2016 (que pode ter dados ainda parciais), o WiiU vem mantendo uma fatia relativamente estável do mercado e as vendas de jogos performam com uma mediana similar (0.22M USD) a seus dois principais concorrentes XOne (0.22M USD) e PS4 (0.20M USD).


### Análisando a correlação entre avaliações e vendas no PS3

Escolhi o PS3 pois tem o maior volume de jogos do nosso intervalo de jogos recentes e mais gerações com grande volume de vendas. Podemos comparar os dados históricos das gerações anteriores e fazer algumas projeções para as vendas no PS4.

Aqui vamos usar nosso conjunto mais amplo de dados, desde 1995.

In [506]:
# Definindo fatias dos dados para as gerações de playstation
df_full_PS1 = df_full[(df_full['platform'] == 'PS')]
df_full_PS2 = df_full[(df_full['platform'] == 'PS2')]
df_full_PS3 = df_full[(df_full['platform'] == 'PS3')]
df_full_PS4 = df_full[(df_full['platform'] == 'PS4')]


In [507]:
# Observando a distribuição de vendas no PS3
fig = px.histogram(df_full_PS3, x='total_sales', nbins=50, title='Distribuição de Vendas Totais no PS3', width=1000, height=600)
fig.update_layout(xaxis_title='Vendas Totais (em milhões USD)', yaxis_title='Número de Jogos')
fig.show()

In [508]:
# Correlação entre critic_score e total_sales

fig = px.scatter(df_full_PS3, x='critic_score', y='total_sales', hover_name='name', color='year_of_release', title='Correlação entre Critic Score e Vendas Totais', width=1400, height=800)
fig.update_layout(xaxis_title='Avaliação dos Críticos', yaxis_title='Vendas Totais (em milhões USD)')
fig.show()

In [509]:
# Correlação entre user_score e total_sales

fig = px.scatter(df_full_PS3, x='user_score', y='total_sales', hover_name='name', color='year_of_release', title='Correlação entre User Score e Vendas Totais (Jogos Lançados a partir de 2010)', width=1400, height=800)
fig.update_layout(xaxis_title='Avaliação dos Usuários', yaxis_title='Vendas Totais (em milhões USD)')
fig.show()

Para o cálculo das correlações, decidi usar o método de Spearman, pois é mais adequado aos dados de vendas, dado que:

- São discretos (valores em milhões com casas decimais limitadas)
- Seguem uma distribuição exponencial decrescente — muitos jogos com vendas baixas e pouquíssimos com vendas altíssimas

In [510]:
# Calculo de correlação entre critic_score, user_score e total_sales usando o método de Spearman
df_full_PS3[['critic_score', 'user_score', 'total_sales']].corr(method='spearman')

,critic_score,user_score,total_sales
critic_score,1.000000,0.593762,0.655914
user_score,0.593762,1.000000,0.314562
total_sales,0.655914,0.314562,1.000000


In [511]:
# Observando as outras gerações
print('Correlação entre critic_score, user_score e total_sales para PS1:')
display(df_full_PS1[['critic_score', 'user_score', 'total_sales']].corr(method='spearman'))
print('Correlação entre critic_score, user_score e total_sales para PS2:')
display(df_full_PS2[['critic_score', 'user_score', 'total_sales']].corr(method='spearman'))
print('Correlação entre critic_score, user_score e total_sales para PS4:')
display(df_full_PS4[['critic_score', 'user_score', 'total_sales']].corr(method='spearman'))

Correlação entre critic_score, user_score e total_sales para PS1:


,critic_score,user_score,total_sales
critic_score,1.000000,0.599562,0.556097
user_score,0.599562,1.000000,0.354225
total_sales,0.556097,0.354225,1.000000


Correlação entre critic_score, user_score e total_sales para PS2:


,critic_score,user_score,total_sales
critic_score,1.000000,0.525645,0.470647
user_score,0.525645,1.000000,0.219998
total_sales,0.470647,0.219998,1.000000


Correlação entre critic_score, user_score e total_sales para PS4:


,critic_score,user_score,total_sales
critic_score,1.000000,0.433309,0.508238
user_score,0.433309,1.000000,-0.028340
total_sales,0.508238,-0.028340,1.000000


In [512]:
# Vamos comparar com alguns concorrentes

df_full_X360 = df_full[(df_full['platform'] == 'X360')]
df_full_Wii = df_full[(df_full['platform'] == 'Wii')]
df_full_DS = df_full[(df_full['platform'] == 'DS')]
df_full_PC = df_full[(df_full['platform'] == 'PC')]

print('Correlação entre critic_score, user_score e total_sales para X360:')
display(df_full_X360[['critic_score', 'user_score', 'total_sales']].corr(method='spearman'))
print('Correlação entre critic_score, user_score e total_sales para Wii:')
display(df_full_Wii[['critic_score', 'user_score', 'total_sales']].corr(method='spearman'))
print('Correlação entre critic_score, user_score e total_sales para DS:')
display(df_full_DS[['critic_score', 'user_score', 'total_sales']].corr(method='spearman'))
print('Correlação entre critic_score, user_score e total_sales para PC:')
display(df_full_PC[['critic_score', 'user_score', 'total_sales']].corr(method='spearman'))

Correlação entre critic_score, user_score e total_sales para X360:


,critic_score,user_score,total_sales
critic_score,1.000000,0.596228,0.660874
user_score,0.596228,1.000000,0.291071
total_sales,0.660874,0.291071,1.000000


Correlação entre critic_score, user_score e total_sales para Wii:


,critic_score,user_score,total_sales
critic_score,1.000000,0.677807,0.429998
user_score,0.677807,1.000000,0.230137
total_sales,0.429998,0.230137,1.000000


Correlação entre critic_score, user_score e total_sales para DS:


,critic_score,user_score,total_sales
critic_score,1.000000,0.616214,0.369944
user_score,0.616214,1.000000,0.206210
total_sales,0.369944,0.206210,1.000000


Correlação entre critic_score, user_score e total_sales para PC:


,critic_score,user_score,total_sales
critic_score,1.000000,0.572127,0.353519
user_score,0.572127,1.000000,-0.009119
total_sales,0.353519,-0.009119,1.000000


#### Observações:

- Parece que há uma correlação moderada entre a avaliação dos criticos e as vendas. Podemos dizer que há uma possibilidade considerável de boas avaliações de críticos impactarem as vendas ou vice-versa. Essa correlação moderada se mantém ao longo das gerações de Playstation, mas é observada de forma mais clara no PS3.

- Já para as avaliações de usuários, existe uma correlação mais fraca com o total de vendas de um jogo. No PS4, não existe correlação entre as avaliações de usuários e as vendas, talvez isso mude ao longo dos anos a medida que o console se estabeleça melhor no mercado. 

- O mesmo pode ser observado para o concorrente direto XBox 360

- Para as plataformas da Nintendo e PC a avaliação dos críticos parece ter menos influência sobre as vendas.


### Análise de vendas cross-plataforma

Para a análise cross-plataforma vamos trabalhar a partir do PS3 e fazer algumas comparações, considerando apenas os jogos que saíram em outras plataformas:

- Como performaram os top 20 jogos mais vendidos em outras plataformas.
- Como os jogos com performance típica (no intervalo q1 a q3) performam em outras plataformas

Vamos comparar principalmente com Xbox360, PC e Wii. Vamos comparar também com as outras gerações: PS2 e PS4

In [513]:
# Criando um dataframe apenas com jogos presentes em mais de uma plataforma, para analisar se jogos multiplataforma têm melhor desempenho em vendas
df_multiplatform = df_full[df_full.duplicated(subset=['name', 'year_of_release'], keep=False)].reset_index(drop=True)
df_multiplatform.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5127 entries, 0 to 5126
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   index            5127 non-null   int64  
 1   name             5127 non-null   object 
 2   platform         5127 non-null   object 
 3   year_of_release  5127 non-null   int64  
 4   genre            5127 non-null   object 
 5   na_sales         5127 non-null   float64
 6   eu_sales         5127 non-null   float64
 7   jp_sales         5127 non-null   float64
 8   other_sales      5127 non-null   float64
 9   critic_score     3248 non-null   float64
 10  user_score       3322 non-null   float64
 11  rating           5127 non-null   object 
 12  total_sales      5127 non-null   float64
dtypes: float64(7), int64(2), object(4)
memory usage: 520.8+ KB


#### Performance dos Top 20 mais vendidos no PS3 em outras plataformas

In [514]:
# Top 20 jogos PS3 com maiores vendas
top_20_ps3 = df_full_PS3.sort_values('total_sales', ascending=False).head(20)
top_20_ps3_list = top_20_ps3['name'].tolist()
print('Top 20 jogos PS3 com maiores vendas:')
display(top_20_ps3[['name', 'platform', 'year_of_release', 'genre', 'total_sales', 'critic_score', 'user_score']])

Top 20 jogos PS3 com maiores vendas:


,name,platform,year_of_release,genre,total_sales,critic_score,user_score
11,Grand Theft Auto V,PS3,2013,Action,21.05,97.0,82.0
25,Call of Duty: Black Ops II,PS3,2012,Shooter,13.79,83.0,53.0
28,Call of Duty: Modern Warfare 3,PS3,2011,Shooter,13.33,88.0,32.0
32,Call of Duty: Black Ops,PS3,2010,Shooter,12.63,88.0,64.0
43,Gran Turismo 5,PS3,2010,Racing,10.70,84.0,75.0
44,Call of Duty: Modern Warfare 2,PS3,2009,Shooter,10.61,94.0,63.0
45,Grand Theft Auto IV,PS3,2008,Action,10.50,98.0,75.0
56,Call of Duty: Ghosts,PS3,2013,Shooter,9.36,71.0,26.0
65,FIFA Soccer 13,PS3,2012,Action,8.17,88.0,66.0
84,Battlefield 3,PS3,2011,Shooter,7.17,85.0,75.0


In [515]:
# Criando uma DataFrame comparando os top 20 jogos do PS3 com suas versões em outras plataformas, para analisar se os jogos mais vendidos do PS3 tiveram melhor desempenho em vendas e scores em relação às suas versões em outras plataformas
df_top_20_multiplatform = df_multiplatform[df_multiplatform['name'].isin(top_20_ps3_list)].reset_index(drop=True)
df_top_20_multiplatform.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   index            70 non-null     int64  
 1   name             70 non-null     object 
 2   platform         70 non-null     object 
 3   year_of_release  70 non-null     int64  
 4   genre            70 non-null     object 
 5   na_sales         70 non-null     float64
 6   eu_sales         70 non-null     float64
 7   jp_sales         70 non-null     float64
 8   other_sales      70 non-null     float64
 9   critic_score     54 non-null     float64
 10  user_score       59 non-null     float64
 11  rating           70 non-null     object 
 12  total_sales      70 non-null     float64
dtypes: float64(7), int64(2), object(4)
memory usage: 7.2+ KB


In [516]:
# Comparando as vendas dos jogos de PS3 com suas versões em outras plataformas usando um gráfico de barras
fig = px.bar(df_top_20_multiplatform, x='name', y='total_sales', color='platform', title='Comparação de Vendas dos Jogos mais vendidos no PS3 com suas Versões em Outras Plataformas', width=1400, height=800)
fig.update_layout(xaxis_title='Jogo', yaxis_title='Vendas Totais (em milhões USD)', xaxis_tickangle=-45)
fig.show()

In [517]:
# Criando um heatmap para comparar as vendas dos jogos de PS3 com suas versões em outras plataformas
fig = px.density_heatmap(df_top_20_multiplatform, x='platform', y='name', z='total_sales', title='Comparação de Vendas dos Jogos mais vendidos no PS3 com suas Versões em Outras Plataformas', color_continuous_scale='Viridis', width=1400, height=800)
fig.update_layout(xaxis_title='Plataforma', yaxis_title='Jogo', coloraxis_colorbar_title='Vendas Totais (em milhões USD)')
fig.show()

#### Comparação geral PS3 v. concorrentes diretos (Xbox360, Wii e PC)

In [551]:
# Criando uma pivot table multiplataforma
pivot_multiplatform = df_multiplatform.pivot_table(index='name', columns='platform', values='total_sales')
display(pivot_multiplatform)
pivot_multiplatform.info()

platform,3DS,DS,GBA,PC,PS,PS2,PS3,PS4,PSP,PSV,Wii,WiiU,X360,XOne
name,,,,,,,,,,,,,,
Frozen: Olaf's Quest,0.59,0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
007: Quantum of Solace,NaN,0.13,NaN,0.02,NaN,0.43,1.15,NaN,NaN,NaN,0.65,NaN,1.48,NaN
2010 FIFA World Cup South Africa,NaN,NaN,NaN,NaN,NaN,NaN,1.23,NaN,0.46,NaN,0.43,NaN,0.85,NaN
2014 FIFA World Cup Brazil,NaN,NaN,NaN,NaN,NaN,NaN,0.61,NaN,NaN,NaN,NaN,NaN,0.43,NaN
3rd Super Robot Wars Z Jigoku Hen,NaN,NaN,NaN,NaN,NaN,NaN,0.23,NaN,NaN,0.19,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
iCarly,NaN,0.72,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.45,NaN,NaN,NaN
iCarly 2: iJoin The Click!,NaN,0.27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.10,NaN,NaN,NaN
nail'd,NaN,NaN,NaN,NaN,NaN,NaN,0.12,NaN,NaN,NaN,NaN,NaN,0.11,NaN


<class 'pandas.core.frame.DataFrame'>
Index: 1780 entries,  Frozen: Olaf's Quest to uDraw Studio: Instant Artist
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   3DS     108 non-null    float64
 1   DS      447 non-null    float64
 2   GBA     140 non-null    float64
 3   PC      404 non-null    float64
 4   PS      28 non-null     float64
 5   PS2     479 non-null    float64
 6   PS3     959 non-null    float64
 7   PS4     303 non-null    float64
 8   PSP     290 non-null    float64
 9   PSV     150 non-null    float64
 10  Wii     585 non-null    float64
 11  WiiU    85 non-null     float64
 12  X360    924 non-null    float64
 13  XOne    223 non-null    float64
dtypes: float64(14)
memory usage: 208.6+ KB


In [558]:
# Criando um gráfico de dispersão para comparar as vendas dos jogos multiplataforma no PS3 com as vendas em outras plataformas
fig = px.scatter(pivot_multiplatform, x='PS3', y='X360', hover_name=pivot_multiplatform.index, title='Comparação de Vendas entre PS3 e X360 para Jogos Multiplataforma', width=900, height=600)
fig.update_layout(xaxis_title='Vendas no PS3 (em milhões USD)', yaxis_title='Vendas no X360 (em milhões USD)')
fig.update_xaxes(range=[0, 22])
fig.update_yaxes(range=[0, 22])
fig.add_shape( # Adicionando uma linha de referência onde as vendas seriam iguais nos dois consoles
    type="line",
    x0=0, y0=0, x1=22, y1=22,
    line=dict(color="Red", width=1, dash="dash")
)
fig.show()

In [585]:
# Contagem de jogos mais vendidos no Xbox360 v. Jogos mais vendidos no PS3
filter_360_PS3 = pivot_multiplatform[['X360', 'PS3']].dropna()
more_X360 = (pivot_multiplatform['X360'] > pivot_multiplatform['PS3']).sum()
more_PS3 = (pivot_multiplatform['PS3'] > pivot_multiplatform['X360']).sum()
sales_360 = filter_360_PS3['X360'].sum()
sales_PS3 = filter_360_PS3['PS3'].sum()
print(f'Número de jogos multiplataforma com vendas maiores no X360 do que no PS3: {more_X360}')
print(f'Número de jogos multiplataforma com vendas maiores no PS3 do que no X360: {more_PS3}')
print(f'Vendas totais dos jogos multiplataforma no X360: {sales_360:.2f} milhões USD')
print(f'Vendas totais dos jogos multiplataforma no PS3: {sales_PS3:.2f} milhões USD')

Número de jogos multiplataforma com vendas maiores no X360 do que no PS3: 319
Número de jogos multiplataforma com vendas maiores no PS3 do que no X360: 444
Vendas totais dos jogos multiplataforma no X360: 681.01 milhões USD
Vendas totais dos jogos multiplataforma no PS3: 685.58 milhões USD


In [583]:
# Criando um gráfico de dispersão para comparar as vendas dos jogos multiplataforma no PS3 com as vendas em outras plataformas
fig = px.scatter(pivot_multiplatform, x='PS3', y='PC', hover_name=pivot_multiplatform.index, title='Comparação de Vendas entre PS3 e PC para Jogos Multiplataforma', width=900, height=600)
fig.update_layout(xaxis_title='Vendas no PS3 (em milhões USD)', yaxis_title='Vendas no PC (em milhões USD)')
fig.update_xaxes(range=[0, 22])
fig.update_yaxes(range=[0, 22])
fig.add_shape( # Adicionando uma linha de referência onde as vendas seriam iguais nas duas plataformas
    type="line",
    x0=0, y0=0, x1=22, y1=22,
    line=dict(color="Red", width=1, dash="dash")
)
fig.show()

In [581]:
# Contagem de jogos mais vendidos no PC v. Jogos mais vendidos no PS3
filter_PC_PS3 = pivot_multiplatform[['PC', 'PS3']].dropna()
more_PC = (filter_PC_PS3['PC'] > filter_PC_PS3['PS3']).sum()
more_PS3 = (filter_PC_PS3['PS3'] > filter_PC_PS3['PC']).sum()
sales_PC = filter_PC_PS3['PC'].sum()
sales_PS3 = filter_PC_PS3['PS3'].sum()
print(f'Número de jogos multiplataforma com vendas maiores no PC do que no PS3: {more_PC}')
print(f'Número de jogos multiplataforma com vendas maiores no PS3 do que no PC: {more_PS3}')
print(f'Vendas totais dos jogos multiplataforma no PC: {sales_PC:.2f} milhões USD')
print(f'Vendas totais dos jogos multiplataforma no PS3: {sales_PS3:.2f} milhões USD')

Número de jogos multiplataforma com vendas maiores no PC do que no PS3: 6
Número de jogos multiplataforma com vendas maiores no PS3 do que no PC: 257
Vendas totais dos jogos multiplataforma no PC: 57.10 milhões USD
Vendas totais dos jogos multiplataforma no PS3: 367.54 milhões USD


In [582]:
# Criando um gráfico de dispersão para comparar as vendas dos jogos multiplataforma no PS3 com as vendas em outras plataformas
fig = px.scatter(pivot_multiplatform, x='PS3', y='Wii', hover_name=pivot_multiplatform.index, title='Comparação de Vendas entre PS3 e Wii para Jogos Multiplataforma', width=900, height=600)
fig.update_layout(xaxis_title='Vendas no PS3 (em milhões USD)', yaxis_title='Vendas no Wii (em milhões USD)')
fig.update_xaxes(range=[0, 22])
fig.update_yaxes(range=[0, 22])
fig.add_shape( # Adicionando uma linha de referência onde as vendas seriam iguais nas duas plataformas
    type="line",
    x0=0, y0=0, x1=22, y1=22,
    line=dict(color="Red", width=1, dash="dash")
)
fig.show()

In [579]:
# Contagem de jogos mais vendidos no Wii v. Jogos mais vendidos no PS3
filter_Wii_PS3 = pivot_multiplatform[pivot_multiplatform['Wii'].notnull() & pivot_multiplatform['PS3'].notnull()]
more_Wii = (filter_Wii_PS3['Wii'] > filter_Wii_PS3['PS3']).sum()
more_PS3 = (filter_Wii_PS3['PS3'] > filter_Wii_PS3['Wii']).sum()
sales_Wii = filter_Wii_PS3['Wii'].sum()
sales_PS3 = filter_Wii_PS3['PS3'].sum()
print(f'Número de jogos multiplataforma com vendas maiores no Wii do que no PS3: {more_Wii}')
print(f'Número de jogos multiplataforma com vendas maiores no PS3 do que no Wii: {more_PS3}')
print(f'Vendas totais dos jogos multiplataforma no Wii: {sales_Wii:.2f} milhões USD')
print(f'Vendas totais dos jogos multiplataforma no PS3: {sales_PS3:.2f} milhões USD')

Número de jogos multiplataforma com vendas maiores no Wii do que no PS3: 150
Número de jogos multiplataforma com vendas maiores no PS3 do que no Wii: 112
Vendas totais dos jogos multiplataforma no Wii: 178.85 milhões USD
Vendas totais dos jogos multiplataforma no PS3: 205.79 milhões USD


#### Comparação dos jogos de PS3 com suas outras gerações (PS2 e PS4)

In [564]:
# Criando um gráfico de dispersão para comparar as vendas dos jogos multiplataforma no PS3 com as vendas em outras plataformas
fig = px.scatter(pivot_multiplatform, x='PS3', y='PS2', hover_name=pivot_multiplatform.index, title='Comparação de Vendas entre PS3 e PS2 para Jogos Multiplataforma', width=900, height=600)
fig.update_layout(xaxis_title='Vendas no PS3 (em milhões USD)', yaxis_title='Vendas no PS2 (em milhões USD)')
fig.update_xaxes(range=[0, 22])
fig.update_yaxes(range=[0, 22])
fig.add_shape( # Adicionando uma linha de referência onde as vendas seriam iguais nos dois consoles
    type="line",
    x0=0, y0=0, x1=22, y1=22,
    line=dict(color="Red", width=1, dash="dash")
)
fig.show()

In [578]:
# Contagem de jogos mais vendidos no PS2 v. Jogos mais vendidos no PS3
filter_PS2_PS3 = pivot_multiplatform[pivot_multiplatform['PS2'].notnull() & pivot_multiplatform['PS3'].notnull()]
more_PS2 = (filter_PS2_PS3['PS2'] > filter_PS2_PS3['PS3']).sum()
more_PS3 = (filter_PS2_PS3['PS3'] > filter_PS2_PS3['PS2']).sum()
sales_PS2 = filter_PS2_PS3['PS2'].sum()
sales_PS3 = filter_PS2_PS3['PS3'].sum()
print(f'Número de jogos multiplataforma com vendas maiores no PS2 do que no PS3: {more_PS2}')
print(f'Número de jogos multiplataforma com vendas maiores no PS3 do que no PS2: {more_PS3}')
print(f'Vendas totais dos jogos multiplataforma no PS2: {sales_PS2:.2f} milhões USD')
print(f'Vendas totais dos jogos multiplataforma no PS3: {sales_PS3:.2f} milhões USD')

Número de jogos multiplataforma com vendas maiores no PS2 do que no PS3: 74
Número de jogos multiplataforma com vendas maiores no PS3 do que no PS2: 82
Vendas totais dos jogos multiplataforma no PS2: 102.97 milhões USD
Vendas totais dos jogos multiplataforma no PS3: 114.72 milhões USD


In [570]:
# Criando um gráfico de dispersão para comparar as vendas dos jogos multiplataforma no PS3 com as vendas em outras plataformas
fig = px.scatter(pivot_multiplatform, x='PS3', y='PS4', hover_name=pivot_multiplatform.index, title='Comparação de Vendas entre PS3 e PS4 para Jogos Multiplataforma', width=900, height=600)
fig.update_layout(xaxis_title='Vendas no PS3 (em milhões USD)', yaxis_title='Vendas no PS4 (em milhões USD)')
fig.update_xaxes(range=[0, 22])
fig.update_yaxes(range=[0, 22])
fig.add_shape( # Adicionando uma linha de referência onde as vendas seriam iguais nos dois consoles
    type="line",
    x0=0, y0=0, x1=22, y1=22,
    line=dict(color="Red", width=1, dash="dash")
)
fig.show()

In [ ]:
# Contagem de jogos mais vendidos no PS4 v. Jogos mais vendidos no PS3
filter_ps4_ps3 = pivot_multiplatform[pivot_multiplatform['PS4'].notnull() & pivot_multiplatform['PS3'].notnull()]
more_PS4 = (filter_ps4_ps3['PS4'] > filter_ps4_ps3['PS3']).sum()
sales_PS4 = filter_ps4_ps3['PS4'].sum()
more_PS3 = (filter_ps4_ps3['PS3'] > filter_ps4_ps3['PS4']).sum()
sales_PS3 = filter_ps4_ps3['PS3'].sum()
print(f'Número de jogos multiplataforma com vendas maiores no PS4 do que no PS3: {more_PS4}')
print(f'Número de jogos multiplataforma com vendas maiores no PS3 do que no PS4: {more_PS3}')
print(f'Quantidade total vendida dos jogos multiplataforma no PS4: {sales_PS4:.2f} milhões USD')
print(f'Quantidade total vendida dos jogos multiplataforma no PS3: {sales_PS3:.2f} milhões USD')

Número de jogos multiplataforma com vendas maiores no PS4 do que no PS3: 114
Número de jogos multiplataforma com vendas maiores no PS3 do que no PS4: 43
Quantidade total vendida dos jogos multiplataforma no PS4: 177.08 milhões USD
Quantidade total vendida dos jogos multiplataforma no PS3: 135.71 milhões USD


#### Comparação dos jogos com performance típica no PS3 com outras plataformas

In [566]:
# Comparando os jogos com performance típica em vendas no PS3 com sua performance em outras plataformas
typical_sales_ps3 = df_full_PS3[(df_full_PS3['total_sales'] > df_full_PS3['total_sales'].quantile(0.25)) & (df_full_PS3['total_sales'] < df_full_PS3['total_sales'].quantile(0.75))]
typical_sales_ps3_list = typical_sales_ps3['name'].tolist()
print('Jogos com performance típica em vendas no PS3:')
display(typical_sales_ps3[['name', 'platform', 'year_of_release', 'genre', 'total_sales', 'critic_score', 'user_score']])

Jogos com performance típica em vendas no PS3:


,name,platform,year_of_release,genre,total_sales,critic_score,user_score
2211,Genji: Days of the Blade,PS3,2006,Action,0.75,55.0,58.0
2216,DJ Hero 2,PS3,2010,Misc,0.76,86.0,78.0
2219,SoulCalibur V,PS3,2012,Fighting,0.74,81.0,67.0
2223,Kane & Lynch: Dead Men,PS3,2007,Shooter,0.75,NaN,NaN
2237,The Lord of the Rings: Conquest,PS3,2009,Action,0.74,54.0,71.0
...,...,...,...,...,...,...,...
8168,SingStar Latino,PS3,2009,Misc,0.12,NaN,NaN
8169,NFL Tour,PS3,2008,Sports,0.12,45.0,NaN
8245,MotoGP 14,PS3,2014,Racing,0.12,NaN,NaN
8329,Bodycount,PS3,2011,Shooter,0.12,50.0,35.0


In [567]:
# Filtrando o DF pelos jogos típicos de PS3
df_typical_multiplatform = df_multiplatform[df_multiplatform['name'].isin(typical_sales_ps3_list)].reset_index(drop=True)
df_typical_multiplatform.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1667 entries, 0 to 1666
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   index            1667 non-null   int64  
 1   name             1667 non-null   object 
 2   platform         1667 non-null   object 
 3   year_of_release  1667 non-null   int64  
 4   genre            1667 non-null   object 
 5   na_sales         1667 non-null   float64
 6   eu_sales         1667 non-null   float64
 7   jp_sales         1667 non-null   float64
 8   other_sales      1667 non-null   float64
 9   critic_score     1151 non-null   float64
 10  user_score       1220 non-null   float64
 11  rating           1667 non-null   object 
 12  total_sales      1667 non-null   float64
dtypes: float64(7), int64(2), object(4)
memory usage: 169.4+ KB


In [568]:
# Criando uma pivot_table para comparar as vendas de jogos multiplataforma em diferentes plataformas
mean_sales_typical_multiplatform = df_typical_multiplatform.groupby(['platform']).agg({'total_sales': 'mean', 'name': 'count'}).reset_index()
mean_sales_typical_multiplatform = mean_sales_typical_multiplatform.sort_values('total_sales', ascending=False).reset_index(drop=True)
mean_sales_typical_multiplatform.info()
display(mean_sales_typical_multiplatform)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   platform     13 non-null     object 
 1   total_sales  13 non-null     float64
 2   name         13 non-null     int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 444.0+ bytes


,platform,total_sales,name
0,PS4,0.914091,66
1,Wii,0.625772,149
2,XOne,0.506939,49
3,PS2,0.404854,103
4,X360,0.374811,424
5,DS,0.367564,78
6,PS3,0.349002,501
7,WiiU,0.325200,25
8,PSP,0.253467,75
9,3DS,0.204545,22


In [569]:
# Criando um grafico de barra para comparar as vendas médias dos jogos típicos de PS3 com suas versões em outras plataformas
fig = px.bar(mean_sales_typical_multiplatform, x='platform', y='total_sales', color='platform', title='Comparação de Vendas Médias dos Jogos Típicos de PS3 com suas Versões em Outras Plataformas', width=1400, height=800)
fig.update_layout(xaxis_title='Plataforma', yaxis_title='Vendas Médias (em milhões USD)')
fig.show()

#### Conclusão da comparação do PS3 com outras plataformas

- O PS3 foi uma plataforma muito relevante em vendas, especialmente no universo multiplataforma. Ele apresenta vendas totais robustas e um grande volume de títulos comparado a outras plataformas. Sua boa performance, nos permite fazer projeções otimistas para o PS4.
- PS3 v. X360:
    - O X360 tem mais jogos com vendas maiores que os do PS3 em multiplataforma.
    - Ainda assim, o total agregado de vendas entre PS3 e X360 é bastante próximo, mostrando que ambos dominaram o mercado nesta geração.
- PS3 v. Wii:
    - O Wii se destaca com boas vendas médias em títulos típicos do PS3, mas o PS3 continua forte em vendas totais e na variedade de jogos multiplataforma.
    - Isso sugere que o PS3 teve maior presença em blockbusters multiplataforma, enquanto o Wii teve desempenho sólido em nichos casuais e exclusivos.
- PS3 v. PC:
    - O PC aparece com vendas médias mais baixas em relação ao PS3 nos títulos multiplataforma analisados.
    - Isso indica que, no dataset disponível, o PC não era a plataforma dominante para esses lançamentos em termos de vendas totais.
- Em relação às gerações de PlayStation:
    - Os jogos de PS2 que também foram lançados para PS3 tiveram boa performance, vendendo bem em ambas plataformas.
    - O PS4 mostra medianas de vendas mais elevadas em jogos típicos de PS3, sinalizando maior potencial da nova geração, embora ainda com biblioteca menor no período considerado. 
    - Os jogos de PS3 que também foram lançados para PS4 tiveram uma ótima performance no PS4, superando o total de vendas no PS3.
    - Apostar na geração nova (PS4 e XboxOne), mas ainda oferecer suporte para usuários da geração antiga, pode ser uma boa estratégia para o próximo ano.

Em suma, o PS3 ainda é uma plataforma competitiva e bem posicionada dentro do ecossistema multiplataforma, apesar da queda considerável de vendas que migraram para o PS4. A análise mostra que a próxima geração (PS4 e também XOne) apresenta sinais mais fortes de crescimento e potencial de lucro em títulos típicos.

### Análise por gênero

Para análise de gênero vamos comparar o df_full, com dados desde 1995, com os dados mais recentes de df_recent, desde 2010, para entender se existem mudanças de tendencia no mercado atual. Se não houver mudança, podemos trabalhar com o volume maior de dados para uma análise mais confiável. Definindo o dataset mais útil para nossa análise, vamos:

- Avaliar quais gêneros são mais lucrativos e menos lucrativos
- Tentar descobrir o que é possível generalizar sobre os genêros mais e menos lucrativos)

In [628]:
# Agrupando o dataframe desde 1995 por gênero para um overview das vendas por gênero
upto_2009 = df_full['year_of_release'] <= 2009
df_genre_sales_1995 = df_full[upto_2009].groupby('genre').agg({'total_sales': ('sum', 'mean', 'median'), 'name': 'count'}).reset_index()
df_genre_sales_1995.columns = ['genre', 'total_sales', 'mean_sales', 'median_sales', 'game_count']
df_genre_sales_1995 = df_genre_sales_1995.sort_values('median_sales', ascending=False).reset_index(drop=True)
df_genre_sales_1995.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   genre         12 non-null     object 
 1   total_sales   12 non-null     float64
 2   mean_sales    12 non-null     float64
 3   median_sales  12 non-null     float64
 4   game_count    12 non-null     int64  
dtypes: float64(3), int64(1), object(1)
memory usage: 612.0+ bytes


In [631]:
display(df_genre_sales_1995)

,genre,total_sales,mean_sales,median_sales,game_count
0,Platform,382.67,0.803929,0.290,476
1,Sports,804.31,0.636321,0.240,1264
2,Racing,471.34,0.661994,0.235,712
3,Action,851.74,0.596039,0.230,1429
4,Fighting,254.42,0.590302,0.230,431
5,Role-Playing,436.55,0.586761,0.190,744
6,Simulation,276.06,0.488602,0.180,565
7,Misc,490.27,0.492239,0.175,996
8,Shooter,389.17,0.631769,0.170,616
9,Puzzle,121.40,0.329891,0.100,368


In [629]:
# Agrupando o dataframe desde 2010 por gênero para um overview das vendas por gênero
df_genre_sales_2010 = df_recent.groupby('genre').agg({'total_sales': ('sum', 'mean', 'median'), 'name': 'count'}).reset_index()
df_genre_sales_2010.columns = ['genre', 'total_sales', 'mean_sales', 'median_sales', 'game_count']
df_genre_sales_2010 = df_genre_sales_2010.sort_values('median_sales', ascending=False).reset_index(drop=True)
df_genre_sales_2010.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   genre         12 non-null     object 
 1   total_sales   12 non-null     float64
 2   mean_sales    12 non-null     float64
 3   median_sales  12 non-null     float64
 4   game_count    12 non-null     int64  
dtypes: float64(3), int64(1), object(1)
memory usage: 612.0+ bytes


In [630]:
display(df_genre_sales_2010)

,genre,total_sales,mean_sales,median_sales,game_count
0,Shooter,479.74,1.170098,0.385,410
1,Platform,119.72,0.782484,0.210,153
2,Sports,328.38,0.572091,0.200,574
3,Fighting,81.59,0.410000,0.180,199
4,Racing,122.68,0.517637,0.170,237
5,Role-Playing,315.28,0.555070,0.150,568
6,Action,673.09,0.450227,0.140,1495
7,Misc,234.56,0.407222,0.130,576
8,Simulation,71.75,0.330645,0.100,217
9,Strategy,35.84,0.210824,0.080,170


In [615]:
# Juntando os dataframes para comparação
df_genre_sales_1995['period'] = '1995-2009'
df_genre_sales_2010['period'] = '2010-2016'
df_genre_sales_1995['games_per_year'] = df_genre_sales_1995['game_count'] / (2009- 1995 + 1)
df_genre_sales_2010['games_per_year'] = df_genre_sales_2010['game_count'] / (2016 - 2010 + 1)
df_genre_comparison = pd.concat([df_genre_sales_1995, df_genre_sales_2010], ignore_index=True)
df_genre_comparison.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   genre           24 non-null     object 
 1   total_sales     24 non-null     float64
 2   mean_sales      24 non-null     float64
 3   median_sales    24 non-null     float64
 4   game_count      24 non-null     int64  
 5   period          24 non-null     object 
 6   games_per_year  24 non-null     float64
dtypes: float64(4), int64(1), object(2)
memory usage: 1.4+ KB


In [619]:
# Criando um gráfico de barras para comparar a contagem de jogos por gênero entre os períodos 1995-2009 e 2010-2016
fig = px.bar(df_genre_comparison, x='genre', y='games_per_year', color='period', barmode='group', title='Comparação da Contagem de Jogos por Gênero entre os Períodos 1995-2009 e 2010-2016', width=1100, height=800)
fig.update_layout(xaxis_title='Gênero', yaxis_title='Número de Jogos', xaxis_tickangle=-45)
fig.show()

In [618]:
# Comparando a performance de vendas por genero comparando as medianas
fig = px.bar(df_genre_comparison, x='genre', y='median_sales', color='period', barmode='group', title='Comparação da Mediana de Vendas por Gênero entre os Períodos 1995-2009 e 2010-2016', width=1100, height=800)
fig.update_layout(xaxis_title='Gênero', yaxis_title='Número de Jogos', xaxis_tickangle=-45)
fig.show()

In [627]:
# Criando um boxplot para comparar a distribuição de vendas por gênero no período de 1995-2009
fig = px.box(df_full[upto_2009], x='genre', y='total_sales', hover_name='name', title='Comparação da Distribuição de Vendas por Gênero no Período 1995-2009', width=800, height=1200)
fig.update_layout(xaxis_title='Gênero', yaxis_title='Vendas Totais (em milhões USD)', xaxis_tickangle=-45)
fig.show()

In [626]:
# Criando um boxplot para comparar a distribuição de vendas por gênero no período de 2010-2016
fig = px.box(df_recent, x='genre', y='total_sales', hover_name='name', title='Comparação da Distribuição de Vendas por Gênero no Período 2010-2016', width=800, height=1200)
fig.update_layout(xaxis_title='Gênero', yaxis_title='Vendas Totais (em milhões USD)', xaxis_tickangle=-45)
fig.show()

#### Observações:

Decidi priorizar as medianas na análise, ja que temos muitos outliers nos dados e volumes de dados muito diferentes por gênero

- Os genêros mais bem sucedidos nos anos mais recentes são:
    1. Shooter - com uma mediana de  U$385.000, em comparação com U$170.000 no período de 1995 a 2009
    2. Plataform - se mantém como um dos gêneros mais relevantes, apesar da queda em receita, com uma mediana de U$210.000 v. U$290.000 no período anterior
    3. Sports - também teve uma queda, mas ainda figura como um dos principais com U$200.000 v. U$240.000
- Action ainda é o genêro com maior volume de receita total e maior volume de jogos lançados e pode se dizer que é um dos gêneros mais populares. Mas típicamente, seus jogos não performam tão bem em vendas:
    - Teve um crescimento expressivo no volume de jogos lançados por ano com: em torno de 95 jogos por ano (de 1995 a 2009) para 213 por ano no período de 2010 a 2016.
    - Um jogo típico de ação gerou em torno de U$140.000 de receita, enquanto antes gerava U$230.000, o que pode ser sintoma de um mercado muito mais competitivo com muitas opções
- A principal mudança de cenário é o sucesso de Shooters, que antes estava na 9ª posição em termos de receita típica gerada por um jogo.
- Os genêros que hoje, tipicamente, performam pior em vendas são:
    - Adventure, Puzzle e Strategy: figurando como gêneros de nicho, com mesmo seus outliers performando abaixo dos grandes sucessos do mercado.
        - Essa tendência se mantém desde 1995.
- Outra diferença também pode ser percebida na diferença de escala dos outliers:
    - Grandes jogos de Sports e Racing produziram receitas muito acima da média, o que podemos atribuir ao sucesso imenso do Wii, com destaque ao Wii Sports com U$82 milhões em receita total, que não se repetiu em anos recentes.

## Categorização por região

Nessa etapa, vamos tentar entender diferenças entre as regiões (America do Norte, Europa, Japão, Outras). Como queremos observar as dinâmicas no mercado atual, vamos seguir utilizando o Dataset de 2010 em diante (`df_recent`). A análise responderá 3 perguntas:

- Quais são as 5 principais plataformas de cada região?
- Os 5 principais gêneros de cada região?
- Como a classificação ESRB afeta vendas em cada região?

### Principais plataformas por região

In [ ]:
# Agregando o DF por plataforma para analisar a performance de vendas na América do Norte por plataforma
df_na_platform = df_recent.groupby('platform').agg({'na_sales': ('sum', 'median'), 'name': 'count'}).reset_index()
df_na_platform.columns = ['platform', 'na_total_sales', 'na_median_sales', 'game_count']
df_na_platform = df_na_platform.sort_values('na_median_sales', ascending=False).reset_index(drop=True)
df_na_platform.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   platform         12 non-null     object 
 1   na_total_sales   12 non-null     float64
 2   na_median_sales  12 non-null     float64
 3   game_count       12 non-null     int64  
dtypes: float64(2), int64(1), object(1)
memory usage: 516.0+ bytes


In [648]:
display(df_na_platform)

,platform,na_total_sales,na_median_sales,game_count
0,X360,334.18,0.16,680
1,XOne,93.12,0.12,247
2,Wii,121.20,0.11,450
3,WiiU,38.10,0.11,147
4,PS3,229.25,0.09,888
5,PS4,108.74,0.06,392
6,DS,59.66,0.05,507
7,3DS,82.65,0.01,512
8,PC,39.07,0.00,479
9,PS2,2.32,0.00,45


In [ ]:
# Calculando a cota de mercado de cada plataforma
total_na_sales = df_na_platform['na_total_sales'].sum()
df_na_platform['market_share'] = df_na_platform['na_total_sales'] / total_na_sales * 100
df_na_platform = df_na_platform.sort_values('market_share', ascending=False).reset_index(drop=True)
display(df_na_platform)

,platform,na_total_sales,na_median_sales,game_count,market_share
0,X360,334.18,0.16,680,29.562725
1,PS3,229.25,0.09,888,20.280252
2,Wii,121.20,0.11,450,10.721774
3,PS4,108.74,0.06,392,9.619519
4,XOne,93.12,0.12,247,8.237719
5,3DS,82.65,0.01,512,7.311506
6,DS,59.66,0.05,507,5.277731
7,PC,39.07,0.00,479,3.456268
8,WiiU,38.10,0.11,147,3.370459
9,PSV,12.47,0.00,429,1.103140


A plataforma X360 tem uma cota de mercado de 29.56% na América do Norte, com vendas totais de 334.18 milhões USD e uma mediana de vendas de 0.16 milhões USD por jogo.
A plataforma PS3 tem uma cota de mercado de 20.28% na América do Norte, com vendas totais de 229.25 milhões USD e uma mediana de vendas de 0.09 milhões USD por jogo.
A plataforma Wii tem uma cota de mercado de 10.72% na América do Norte, com vendas totais de 121.20 milhões USD e uma mediana de vendas de 0.11 milhões USD por jogo.
A plataforma PS4 tem uma cota de mercado de 9.62% na América do Norte, com vendas totais de 108.74 milhões USD e uma mediana de vendas de 0.06 milhões USD por jogo.
A plataforma XOne tem uma cota de mercado de 8.24% na América do Norte, com vendas totais de 93.12 milhões USD e uma mediana de vendas de 0.12 milhões USD por jogo.


In [651]:
# Agregando o DF por plataforma para analisar a performance de vendas na Europa por plataforma
df_eu_platform = df_recent.groupby('platform').agg({'eu_sales': ('sum', 'median'), 'name': 'count'}).reset_index()
df_eu_platform.columns = ['platform', 'eu_total_sales', 'eu_median_sales', 'game_count']
df_eu_platform = df_eu_platform.sort_values('eu_median_sales', ascending=False).reset_index(drop=True)
df_eu_platform.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   platform         12 non-null     object 
 1   eu_total_sales   12 non-null     float64
 2   eu_median_sales  12 non-null     float64
 3   game_count       12 non-null     int64  
dtypes: float64(2), int64(1), object(1)
memory usage: 516.0+ bytes


In [652]:
display(df_eu_platform)

,platform,eu_total_sales,eu_median_sales,game_count
0,PS4,141.09,0.08,392
1,X360,163.41,0.08,680
2,WiiU,25.13,0.07,147
3,XOne,51.59,0.07,247
4,PC,68.82,0.05,479
5,PS3,213.59,0.05,888
6,Wii,65.91,0.02,450
7,3DS,61.27,0.00,512
8,DS,28.06,0.00,507
9,PS2,1.67,0.00,45


In [659]:
# Calculando a cota de mercado de cada plataforma
total_eu_sales = df_eu_platform['eu_total_sales'].sum()
df_eu_platform['market_share'] = df_eu_platform['eu_total_sales'] / total_eu_sales * 100
df_eu_platform = df_eu_platform.sort_values('market_share', ascending=False).reset_index(drop=True)
display(df_eu_platform)

,platform,eu_total_sales,eu_median_sales,game_count,market_share
0,PS3,213.59,0.05,888,25.391108
1,X360,163.41,0.08,680,19.425820
2,PS4,141.09,0.08,392,16.772468
3,PC,68.82,0.05,479,8.181170
4,Wii,65.91,0.02,450,7.835235
5,3DS,61.27,0.00,512,7.283642
6,XOne,51.59,0.07,247,6.132905
7,DS,28.06,0.00,507,3.335711
8,WiiU,25.13,0.07,147,2.987399
9,PSV,13.07,0.00,429,1.553733


In [653]:
# Agregando o DF por plataforma para analisar a performance de vendas no Japão por plataforma
df_jp_platform = df_recent.groupby('platform').agg({'jp_sales': ('sum', 'median'), 'name': 'count'}).reset_index()
df_jp_platform.columns = ['platform', 'jp_total_sales', 'jp_median_sales', 'game_count']
df_jp_platform = df_jp_platform.sort_values('jp_median_sales', ascending=False).reset_index(drop=True)
df_jp_platform.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   platform         12 non-null     object 
 1   jp_total_sales   12 non-null     float64
 2   jp_median_sales  12 non-null     float64
 3   game_count       12 non-null     int64  
dtypes: float64(2), int64(1), object(1)
memory usage: 516.0+ bytes


In [654]:
display(df_jp_platform)

,platform,jp_total_sales,jp_median_sales,game_count
0,3DS,100.62,0.05,512
1,PSP,42.20,0.03,500
2,PSV,21.84,0.03,429
3,PS3,59.26,0.02,888
4,PS2,0.80,0.01,45
5,PS4,15.96,0.01,392
6,DS,27.90,0.00,507
7,PC,0.00,0.00,479
8,Wii,17.75,0.00,450
9,WiiU,13.01,0.00,147


In [660]:
# Calculando a cota de mercado de cada plataforma
total_jp_sales = df_jp_platform['jp_total_sales'].sum()
df_jp_platform['market_share'] = df_jp_platform['jp_total_sales'] / total_jp_sales * 100
df_jp_platform = df_jp_platform.sort_values('market_share', ascending=False).reset_index(drop=True)
display(df_jp_platform)

,platform,jp_total_sales,jp_median_sales,game_count,market_share
0,3DS,100.62,0.05,512,32.975028
1,PS3,59.26,0.02,888,19.420594
2,PSP,42.20,0.03,500,13.829718
3,DS,27.90,0.00,507,9.143344
4,PSV,21.84,0.03,429,7.157370
5,Wii,17.75,0.00,450,5.817002
6,PS4,15.96,0.01,392,5.230386
7,WiiU,13.01,0.00,147,4.263617
8,X360,5.46,0.00,680,1.789343
9,PS2,0.80,0.01,45,0.262175


In [655]:
# Agregando o DF por plataforma para analisar a performance de vendas em outras regiões por plataforma
df_other_platform = df_recent.groupby('platform').agg({'other_sales': ('sum', 'median'), 'name': 'count'}).reset_index()
df_other_platform.columns = ['platform', 'other_total_sales', 'other_median_sales', 'game_count']
df_other_platform = df_other_platform.sort_values('other_median_sales', ascending=False).reset_index(drop=True)
df_other_platform.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   platform            12 non-null     object 
 1   other_total_sales   12 non-null     float64
 2   other_median_sales  12 non-null     float64
 3   game_count          12 non-null     int64  
dtypes: float64(2), int64(1), object(1)
memory usage: 516.0+ bytes


In [656]:
display(df_other_platform)

,platform,other_total_sales,other_median_sales,game_count
0,PS3,85.63,0.03,888
1,PS4,48.35,0.03,392
2,WiiU,5.95,0.02,147
3,X360,47.36,0.02,680
4,XOne,14.27,0.02,247
5,DS,8.13,0.01,507
6,PC,14.07,0.01,479
7,Wii,18.11,0.01,450
8,3DS,13.27,0.00,512
9,PS2,1.30,0.00,45


In [661]:
# Calculando a cota de mercado de cada plataforma
total_other_sales = df_other_platform['other_total_sales'].sum()
df_other_platform['market_share'] = df_other_platform['other_total_sales'] / total_other_sales * 100
df_other_platform = df_other_platform.sort_values('market_share', ascending=False).reset_index(drop=True)
display(df_other_platform)

,platform,other_total_sales,other_median_sales,game_count,market_share
0,PS3,85.63,0.03,888,32.013608
1,PS4,48.35,0.03,392,18.076118
2,X360,47.36,0.02,680,17.705997
3,Wii,18.11,0.01,450,6.770600
4,XOne,14.27,0.02,247,5.334978
5,PC,14.07,0.01,479,5.260206
6,3DS,13.27,0.00,512,4.961119
7,DS,8.13,0.01,507,3.039480
8,PSV,6.43,0.00,429,2.403918
9,WiiU,5.95,0.02,147,2.224465


In [ ]:
# Formatando as descobertas com loops de print
print('América do Norte:')
for index, row in df_na_platform.head(5).iterrows():
    print(f"{index + 1}.{row['platform']} tem uma cota de mercado de {row['market_share']:.2f}% ")
    print(f"\tVendas totais de {row['na_total_sales']:.2f} milhões USD e uma mediana de vendas de {row['na_median_sales']:.2f} milhões USD por jogo.")

print('\n\nEuropa:')
for index, row in df_eu_platform.head(5).iterrows():
    print(f"{index + 1}.A plataforma {row['platform']} tem uma cota de mercado de {row['market_share']:.2f}% ")
    print(f"\tVendas totais de {row['eu_total_sales']:.2f} milhões USD e uma mediana de vendas de {row['eu_median_sales']:.2f} milhões USD por jogo.")

print('\n\nJapão:')
for index, row in df_jp_platform.head(5).iterrows():
    print(f"{index + 1}.A plataforma {row['platform']} tem uma cota de mercado de {row['market_share']:.2f}% ")
    print(f"\tVendas totais de {row['jp_total_sales']:.2f} milhões USD e uma mediana de vendas de {row['jp_median_sales']:.2f} milhões USD por jogo.")

print('\n\nOutras Regiões:')
for index, row in df_other_platform.head(5).iterrows():
    print(f"{index + 1}.A plataforma {row['platform']} tem uma cota de mercado de {row['market_share']:.2f}% ")
    print(f"\tVendas totais de {row['other_total_sales']:.2f} milhões USD e uma mediana de vendas de {row['other_median_sales']:.2f} milhões USD por jogo.")

América do Norte:
1.A plataforma X360 tem uma cota de mercado de 29.56% 
	Vendas totais de 334.18 milhões USD e uma mediana de vendas de 0.16 milhões USD por jogo.
2.A plataforma PS3 tem uma cota de mercado de 20.28% 
	Vendas totais de 229.25 milhões USD e uma mediana de vendas de 0.09 milhões USD por jogo.
3.A plataforma Wii tem uma cota de mercado de 10.72% 
	Vendas totais de 121.20 milhões USD e uma mediana de vendas de 0.11 milhões USD por jogo.
4.A plataforma PS4 tem uma cota de mercado de 9.62% 
	Vendas totais de 108.74 milhões USD e uma mediana de vendas de 0.06 milhões USD por jogo.
5.A plataforma XOne tem uma cota de mercado de 8.24% 
	Vendas totais de 93.12 milhões USD e uma mediana de vendas de 0.12 milhões USD por jogo.


Europa:
1.A plataforma PS3 tem uma cota de mercado de 25.39% 
	Vendas totais de 213.59 milhões USD e uma mediana de vendas de 0.05 milhões USD por jogo.
2.A plataforma X360 tem uma cota de mercado de 19.43% 
	Vendas totais de 163.41 milhões USD e uma media

#### Observações:

Abaixo alguns destaques sobre cada região, as cotas e números detalhados podem ser conferidos acima.

**América do Norte:**

- Em termos de percentual de mercado, as top 5 plataformas são, em ordem:
    - Xbox 360, Playstation 3, Wii, Playstation 4 e Xbox One. 
    - Ou seja, a geração atual e a nova geração de consoles da Sony (29,9%) e Microsoft (37,8%) dominam o mercado,
- O WiiU ainda tem uma fatia muito pequena do mercado (quase 3%), mas mantém a mediana de 110.000USD do Wii, indicando um público fiel que segue investido no ecossistema.

**Europa:**
- As top 5 plataformas, em cota de mercado, são:
    - Playstation 3, Xbox 360, Playstation 4, PC, Wii
    - Os consoles do sistema Playstation detém uma fatia maior do mercado na europa (42,16%) em relação a Microsoft (25,56% - entre Xbox360 e XboxOne)
    - O PC aparece como sistema sólido no mercado europeu
- Em termos de mediana, o Wii e os sistemas Microsoft se apresentam com bons indicadores pro mercado europeu

**Japão:**
- As top 5 plataformas no Japão são:
    - 3DS, Playstation 3, Playstation Portable (PSP), DS, Playstion Vita (PSV)
- No mercado japonês dominam os portáteis sob os consoles de mesa, com o PS3 como o único entre os top 5. 

**Outras regiões:**
- Nas outras regiões a lista dos top 5 é a mesma da América do Norte, com algumas varoações de posição.
- Os sistemas Sony dominam as primeiras posições com 50% do mercado, enquanto os sistemas microsoft detém em torno de 23%.

### Principais gêneros por região

In [674]:
# Agregando por gênero e região para analisar a performance de vendas por gênero em cada região
df_genre_region = df_recent.groupby(['genre']).agg({'na_sales': ('sum', 'median'), 'eu_sales': ('sum', 'median'), 'jp_sales': ('sum', 'median'), 'other_sales': ('sum', 'median'), 'name': 'count'}).reset_index()
df_genre_region.columns = ['genre', 'na_total_sales', 'na_median_sales', 'eu_total_sales', 'eu_median_sales', 'jp_total_sales', 'jp_median_sales', 'other_total_sales', 'other_median_sales', 'game_count']
display(df_genre_region)

,genre,na_total_sales,na_median_sales,eu_total_sales,eu_median_sales,jp_total_sales,jp_median_sales,other_total_sales,other_median_sales,game_count
0,Action,290.64,0.050,233.63,0.03,72.20,0.00,76.62,0.01,1495
1,Adventure,20.84,0.000,18.88,0.00,15.67,0.02,5.61,0.00,563
2,Fighting,39.05,0.070,20.33,0.03,13.90,0.02,8.31,0.01,199
3,Misc,123.80,0.060,66.09,0.01,24.29,0.00,20.38,0.01,576
4,Platform,54.90,0.100,38.30,0.08,15.81,0.00,10.71,0.02,153
5,Puzzle,9.10,0.020,6.58,0.02,3.40,0.00,1.52,0.01,114
6,Racing,46.11,0.070,54.75,0.06,6.68,0.00,15.14,0.02,237
7,Role-Playing,112.05,0.015,75.48,0.01,103.54,0.06,24.21,0.01,568
8,Shooter,237.47,0.170,171.45,0.15,14.04,0.00,56.78,0.04,410
9,Simulation,26.39,0.040,26.39,0.02,13.30,0.00,5.67,0.01,217


In [671]:
# Plotando um gráfico de barras para comparar as vendas totais por gênero em cada região
fig = px.bar(df_genre_region, x='genre', y=['na_total_sales', 'eu_total_sales', 'jp_total_sales', 'other_total_sales'], barmode='group', title='Comparação de Vendas Totais por Gênero em Cada Região', width=1400, height=800)
fig.update_layout(xaxis_title='Gênero', yaxis_title='Vendas Totais (em milhões USD)', xaxis_tickangle=-45)
fig.show()

In [672]:
# Plotando um gráfico de barras para comparar a mediana por gênero em cada região
fig = px.bar(df_genre_region, x='genre', y=['na_median_sales', 'eu_median_sales', 'jp_median_sales', 'other_median_sales'], barmode='group', title='Comparação de Vendas Totais por Gênero em Cada Região', width=1400, height=800)
fig.update_layout(xaxis_title='Gênero', yaxis_title='Vendas Totais (em milhões USD)', xaxis_tickangle=-45)
fig.show()

In [675]:
# Criando listas dos 5 principais gêneros por região

# Genêros mais lucrativos (por mediana)
top_genres_na = df_genre_region.sort_values('na_median_sales', ascending=False).head(5)['genre'].tolist()
top_genres_eu = df_genre_region.sort_values('eu_median_sales', ascending=False).head(5)['genre'].tolist()
top_genres_jp = df_genre_region.sort_values('jp_median_sales', ascending=False).head(5)['genre'].tolist()
top_genres_other = df_genre_region.sort_values('other_median_sales', ascending=False).head(5)['genre'].tolist()

# Genêros mais populares (por total de vendas)
top_genres_na_total = df_genre_region.sort_values('na_total_sales', ascending=False).head(5)['genre'].tolist()
top_genres_eu_total = df_genre_region.sort_values('eu_total_sales', ascending=False).head(5)['genre'].tolist()
top_genres_jp_total = df_genre_region.sort_values('jp_total_sales', ascending=False).head(5)['genre'].tolist()
top_genres_other_total = df_genre_region.sort_values('other_total_sales', ascending=False).head(5)['genre'].tolist()

In [ ]:
# Imprimindo as listas - Europa e América do Norte

print('Top 5 gêneros mais lucrativos na América do Norte:')
i = 1
for genre in top_genres_na:
    print(f'{i} - {genre} - Mediana de vendas: {df_genre_region[df_genre_region["genre"] == genre]["na_median_sales"].values[0]:.2f} milhões USD')
    i += 1

print('\nTop 5 gêneros mais populares na América do Norte:')
i = 1
for genre in top_genres_na_total:
    print(f'{i} - {genre} - Total de vendas: {df_genre_region[df_genre_region["genre"] == genre]["na_total_sales"].values[0]:.2f} milhões USD')
    i += 1

print('\nTop 5 gêneros mais lucrativos na Europa:')
i = 1
for genre in top_genres_eu:
    print(f'{i} - {genre} - Mediana de vendas: {df_genre_region[df_genre_region["genre"] == genre]["eu_median_sales"].values[0]:.2f} milhões USD')
    i += 1

print('\nTop 5 gêneros mais populares na Europa:')
i = 1
for genre in top_genres_eu_total:
    print(f'{i} - {genre} - Total de vendas: {df_genre_region[df_genre_region["genre"] == genre]["eu_total_sales"].values[0]:.2f} milhões USD')
    i += 1

Top 5 gêneros mais lucrativos na América do Norte:
1 - Shooter - Mediana de vendas: 0.17 milhões USD
2 - Platform - Mediana de vendas: 0.10 milhões USD
3 - Sports - Mediana de vendas: 0.09 milhões USD
4 - Fighting - Mediana de vendas: 0.07 milhões USD
5 - Racing - Mediana de vendas: 0.07 milhões USD

Top 5 gêneros mais populares na América do Norte:
1 - Action - Total de vendas: 290.64 milhões USD
2 - Shooter - Total de vendas: 237.47 milhões USD
3 - Sports - Total de vendas: 156.81 milhões USD
4 - Misc - Total de vendas: 123.80 milhões USD
5 - Role-Playing - Total de vendas: 112.05 milhões USD

Top 5 gêneros mais lucrativos na Europa:
1 - Shooter - Mediana de vendas: 0.15 milhões USD
2 - Platform - Mediana de vendas: 0.08 milhões USD
3 - Racing - Mediana de vendas: 0.06 milhões USD
4 - Sports - Mediana de vendas: 0.04 milhões USD
5 - Action - Mediana de vendas: 0.03 milhões USD

Top 5 gêneros mais populares na Europa:
1 - Action - Total de vendas: 233.63 milhões USD
2 - Shooter - Tota

In [683]:
# Imprimindo as listas - Japão e Outras regiões

print('Top 5 gêneros mais lucrativos no Japão:')
i = 1
for genre in top_genres_jp:
    print(f'{i} - {genre} - Mediana de vendas: {df_genre_region[df_genre_region["genre"] == genre]["jp_median_sales"].values[0]:.2f} milhões USD')
    i += 1

print('\nTop 5 gêneros mais populares no Japão:')
i = 1
for genre in top_genres_jp_total:
    print(f'{i} - {genre} - Total de vendas: {df_genre_region[df_genre_region["genre"] == genre]["jp_total_sales"].values[0]:.2f} milhões USD')
    i += 1

print('\nTop 5 gêneros mais lucrativos em Outras regiões:')
i = 1
for genre in top_genres_other:
    print(f'{i} - {genre} - Mediana de vendas: {df_genre_region[df_genre_region["genre"] == genre]["other_median_sales"].values[0]:.2f} milhões USD')
    i += 1

print('\nTop 5 gêneros mais populares em Outras regiões:')
i = 1
for genre in top_genres_other_total:
    print(f'{i} - {genre} - Total de vendas: {df_genre_region[df_genre_region["genre"] == genre]["other_total_sales"].values[0]:.2f} milhões USD')
    i += 1

Top 5 gêneros mais lucrativos no Japão:
1 - Role-Playing - Mediana de vendas: 0.06 milhões USD
2 - Adventure - Mediana de vendas: 0.02 milhões USD
3 - Fighting - Mediana de vendas: 0.02 milhões USD
4 - Action - Mediana de vendas: 0.00 milhões USD
5 - Misc - Mediana de vendas: 0.00 milhões USD

Top 5 gêneros mais populares no Japão:
1 - Role-Playing - Total de vendas: 103.54 milhões USD
2 - Action - Total de vendas: 72.20 milhões USD
3 - Misc - Total de vendas: 24.29 milhões USD
4 - Platform - Total de vendas: 15.81 milhões USD
5 - Adventure - Total de vendas: 15.67 milhões USD

Top 5 gêneros mais lucrativos em Outras regiões:
1 - Shooter - Mediana de vendas: 0.04 milhões USD
2 - Platform - Mediana de vendas: 0.02 milhões USD
3 - Racing - Mediana de vendas: 0.02 milhões USD
4 - Sports - Mediana de vendas: 0.02 milhões USD
5 - Action - Mediana de vendas: 0.01 milhões USD

Top 5 gêneros mais populares em Outras regiões:
1 - Action - Total de vendas: 76.62 milhões USD
2 - Shooter - Total d

#### Conclusão — Gênero por região

- NA e Europa: perfis semelhantes — shooters e platform destacam-se pela mediana; Action, Shooter e Sports lideram em volume total. Mercado competitivo: poucos blockbusters impulsionam totals, gêneros nicho entregam melhor retorno típico.
- Japão: perfil distinto — Role‑Playing domina por mediana e total; Adventure e Fighting também fortes. Preferência por narrativas e jogos de luta.
- Outras regiões: padrão próximo a NA/EU — Shooter, Platform e Racing entre os mais lucrativos; Action, Shooter e Sports entre os mais populares.

- Recomendações estratégicas:
    - NA/EU: priorizar shooters, platform e sports; manter investimento em grandes franquias de Action.
    - Japão: focar em RPGs, adventure e fighting; explorar narrativas e IPs regionais.
    - Outras regiões: balancear portfólio entre shooters e plataformas, sem descartar action/sports.
- Estratégia geral: adotar portfólio regionalizado para maximizar retorno médio, mantendo suporte a grandes franquias globais.


### A classificação ESRB afeta vendas por região?

In [684]:
# Quantos jogos existem para cada classificação
print('Jogos por classificação etária:', df_recent['rating'].value_counts())

Jogos por classificação etária: rating
N/A     2164
E        937
T        847
M        719
E10+     603
EC         5
RP         1
Name: count, dtype: int64


In [689]:
# Descartando os jogos sem classificação, EC e RP para a análise
df_rated = df_recent[~df_recent['rating'].isin(['EC', 'RP', 'N/A'])].reset_index(drop=True)


In [690]:
# Agrupando por classificação e região
df_grouped_rating = df_rated.groupby('rating').agg({'na_sales': ('sum', 'median'), 'eu_sales': ('sum', 'median'), 'jp_sales': ('sum', 'median'), 'other_sales': ('sum', 'median'), 'name': 'count'}).reset_index()
df_grouped_rating.columns = ['rating', 'na_total_sales', 'na_median_sales', 'eu_total_sales', 'eu_median_sales', 'jp_total_sales', 'jp_median_sales', 'other_total_sales', 'other_median_sales', 'game_count']
display(df_grouped_rating)

,rating,na_total_sales,na_median_sales,eu_total_sales,eu_median_sales,jp_total_sales,jp_median_sales,other_total_sales,other_median_sales,game_count
0,E,271.15,0.09,197.33,0.04,47.87,0.0,60.04,0.02,937
1,E10+,159.38,0.12,99.28,0.07,13.22,0.0,31.15,0.02,603
2,M,382.22,0.17,292.04,0.14,30.45,0.0,96.98,0.04,719
3,T,160.81,0.09,113.08,0.04,42.11,0.0,38.72,0.02,847


In [692]:
# Criando um gráfico de barras para comparar as vendas totais por classificação e região
fig = px.bar(df_grouped_rating, x='rating', y=['na_total_sales', 'eu_total_sales', 'jp_total_sales', 'other_total_sales'], barmode='group', title='Comparação de Vendas Totais por Classificação e Região', width=1000, height=800)
fig.update_layout(xaxis_title='Classificação', yaxis_title='Vendas Totais (em milhões USD)')
fig.show()

In [693]:
# Adicionando percentual de vendas por região por classificação aos dados
df_grouped_rating['na_sales_percent'] = df_grouped_rating['na_total_sales'] / df_grouped_rating['na_total_sales'].sum() * 100
df_grouped_rating['eu_sales_percent'] = df_grouped_rating['eu_total_sales'] / df_grouped_rating['eu_total_sales'].sum() * 100
df_grouped_rating['jp_sales_percent'] = df_grouped_rating['jp_total_sales'] / df_grouped_rating['jp_total_sales'].sum() * 100
df_grouped_rating['other_sales_percent'] = df_grouped_rating['other_total_sales'] / df_grouped_rating['other_total_sales'].sum() * 100
display(df_grouped_rating)

,rating,na_total_sales,na_median_sales,eu_total_sales,eu_median_sales,jp_total_sales,jp_median_sales,other_total_sales,other_median_sales,game_count,na_sales_percent,eu_sales_percent,jp_sales_percent,other_sales_percent
0,E,271.15,0.09,197.33,0.04,47.87,0.0,60.04,0.02,937,27.851391,28.120502,35.817434,26.462162
1,E10+,159.38,0.12,99.28,0.07,13.22,0.0,31.15,0.02,603,16.370845,14.147892,9.891508,13.729120
2,M,382.22,0.17,292.04,0.14,30.45,0.0,96.98,0.04,719,39.260035,41.617146,22.783389,42.743180
3,T,160.81,0.09,113.08,0.04,42.11,0.0,38.72,0.02,847,16.517729,16.114460,31.507669,17.065538


In [ ]:
# Criando um gráfico com os percentuais para comparar a distribuição de vendas por classificação e região
fig = px.bar(df_grouped_rating, x='rating', y=['na_sales_percent', 'eu_sales_percent', 'jp_sales_percent', 'other_sales_percent'], barmode='stack', title='Comparação de Percentual de Vendas por Classificação e Região', width=1000, height=800)
fig.update_layout(xaxis_title='Classificação', yaxis_title='Percentual de Vendas (%)')
fig.show()

In [702]:
# Criando um gráfico de barras para comparar as medianas por classificação e região
fig = px.bar(df_grouped_rating, x='rating', y=['na_median_sales', 'eu_median_sales', 'jp_median_sales', 'other_median_sales'], barmode='group', title='Comparação de Vendas Medianas por Classificação e Região', width=1000, height=800)
fig.update_layout(xaxis_title='Classificação', yaxis_title='Vendas Medianas (em milhões USD)')
fig.show()

#### Observações:

América do Norte:
  - Classificação com maior receita total: M (382.22 milhões USD)
  - Classificação com maior mediana de vendas: M (0.17 milhões USD)

Europa:
  - Classificação com maior receita total: M (292.04 milhões USD)
  - Classificação com maior mediana de vendas: M (0.14 milhões USD)

Japão:
  - Classificação com maior receita total: E (47.87 milhões USD)
  - Classificação com maior mediana de vendas: E (0.00 milhões USD)

Outras Regiões:
  - Classificação com maior receita total: M (96.98 milhões USD)
  - Classificação com maior mediana de vendas: M (0.04 milhões USD)

Conclusão geral:
  - No agregado, a classificação com maior soma de vendas totais é M.
  - A classificação com maior mediana média de vendas é M.
  - O mercado Japonês tem preferência por títulos com classificação E e T.
  - Isso indica que algumas faixas etárias dominam o mercado em volume, enquanto outras podem entregar maior receita típica por título.
  - A estratégia de portfólio deve considerar tanto o peso de vendas totais quanto o desempenho típico por classificação.

### Conclusão - perfis por região:

**América do Norte**

- Jogador mais alinhado ao ecossistema Xbox, com forte presença de Xbox 360 e Xbox One.
- Prefere especialmente Shooter, Action e Sports.
- Tem alta propensão a consumir títulos com classificação **M (Mature)**.

**Europa**

- Perfil muito próximo ao norte-americano, mas com maior equilíbrio entre PlayStation e Xbox.
- Shooter, Action e Sports concentram o maior volume de vendas.
- Também apresenta forte preferência por títulos classificados como **M (Mature)**.

**Japão**

- Jogador com preferências mais distintas e menos influenciado pelos padrões ocidentais.
- Forte afinidade por plataformas japonesas, especialmente da Sony e Nintendo.
- Role-Playing é o gênero dominante, seguido por Adventure e Fighting.
- Jogos com classificação **E (Everyone)** apresentam melhor desempenho relativo.

**Outras Regiões**

- Perfil semelhante ao da América do Norte e Europa, porém com menor escala de mercado.
- Shooter, Platform e Racing aparecem entre os gêneros mais relevantes.
- PlayStation possui forte presença, acompanhada por Xbox e Nintendo.
- Classificações **M (Mature)** e **E (Everyone)** concentram a maior parte das vendas.

## Testes de hipótese

### Hipótese: As classificações médias dos usuários das plataformas Xbox One e PC são as mesmas.

**Hipóteses:**

* **H0**: A classificações médias dos usuários da plataformas Xbox One é a mesma dos usuários de PC.
* **H1**: A classificações médias dos usuários da plataformas Xbox One é a diferente da classificação média dos usuários de PC.
* Utilizaremos alpha 0.05 por ser um padrão amplamente aceito (e as observações anteriores indicarem uma diferença expressiva nas médias)

Vamos utilizar o teste de hipótese sobre a igualdade da média de duas populações, já que estamos comparando a média de dois grupos. Temos uma diferença relativa entre o tamanho da amostragem das populações, mas variâncias próximas (verificado abaixo), por isso vamos usar o teste t de Student.

In [712]:
# Criando os subsets
df_user_score_xone = df_recent[(df_recent['user_score'].notnull()) & (df_recent['platform'] == 'XOne')]
df_user_score_pc = df_recent[(df_recent['user_score'].notnull()) & (df_recent['platform'] == 'PC')]

display(df_user_score_xone.describe())
display(df_user_score_pc.describe())

print('Variância do user_score para jogos de XOne:', df_user_score_xone['user_score'].var())
print('Variância do user_score para jogos de PC:', df_user_score_pc['user_score'].var())

print()
print('Avaliação média dos usuários para jogos de XOne:', df_user_score_xone['user_score'].mean())
print('Avaliação média dos usuários para jogos de PC:', df_user_score_pc['user_score'].mean())

,index,year_of_release,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,total_sales
count,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,165.000000,182.000000,182.000000
mean,6957.521978,2014.934066,0.431923,0.238132,0.001593,0.066044,73.618182,65.214286,0.737692
std,5326.625596,0.989472,0.620657,0.377369,0.005679,0.092520,12.883849,13.809406,1.018062
min,165.000000,2013.000000,0.000000,0.000000,0.000000,0.000000,20.000000,16.000000,0.010000
25%,1950.750000,2014.000000,0.040000,0.020000,0.000000,0.010000,67.000000,58.000000,0.082500
50%,5887.500000,2015.000000,0.170000,0.090000,0.000000,0.030000,76.000000,68.000000,0.300000
75%,11527.250000,2016.000000,0.557500,0.275000,0.000000,0.090000,83.000000,75.000000,1.057500
max,16660.000000,2016.000000,3.220000,2.190000,0.040000,0.480000,97.000000,92.000000,5.470000


,index,year_of_release,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,total_sales
count,374.000000,374.000000,374.000000,374.000000,374.0,374.000000,348.000000,374.000000,374.000000
mean,9798.387701,2012.457219,0.097059,0.166818,0.0,0.034893,74.537356,65.181818,0.298770
std,4672.627499,2.036408,0.264923,0.284241,0.0,0.075282,11.099824,15.653901,0.582147
min,192.000000,2010.000000,0.000000,0.000000,0.0,0.000000,33.000000,14.000000,0.010000
25%,6141.500000,2011.000000,0.000000,0.030000,0.0,0.000000,68.000000,56.000000,0.040000
50%,10382.500000,2012.000000,0.010000,0.060000,0.0,0.010000,76.000000,68.000000,0.110000
75%,13969.250000,2014.000000,0.090000,0.167500,0.0,0.030000,83.000000,77.000000,0.280000
max,16702.000000,2016.000000,2.570000,2.160000,0.0,0.600000,96.000000,93.000000,5.140000


Variância do user_score para jogos de XOne: 190.69968429360702
Variância do user_score para jogos de PC: 245.0446015110889

Avaliação média dos usuários para jogos de XOne: 65.21428571428571
Avaliação média dos usuários para jogos de PC: 65.18181818181819


In [ ]:
# Testando as hipóteses

# Definindo alpha
alpha = 0.05

# Criando os grupos a serem testados
score_xone = df_user_score_xone['user_score']
score_pc = df_user_score_pc['user_score']

# Realizando o test t de Student para comparar as médias de receita entre os dois planos
statistics, p_value = stats.ttest_ind(score_xone, score_pc, equal_var=True)

# Imprimindo os resultados
print(f'Estatística: {statistics}')
print(f'P-valor: {p_value}')
print(f'Significância: {alpha}')

# Interpretando os resultados
if p_value < alpha:
    print('Rejeitamos a hipótese nula: há uma diferença estatisticamente significativa entre as médias das avaliações de XOne e PC.')
else:    
    print('Não rejeitamos a hipótese nula: não há uma diferença estatisticamente significativa entre as médias das avaliações de XOne e PC.')

Estatística: 0.02382834476454068
P-valor: 0.9809981106490413
Significância: 0.05
Não rejeitamos a hipótese nula: não há uma diferença estatisticamente significativa entre as médias das avaliações de XOne e PC.


### Hipótese: As classificações médias de usuários para os gêneros Action (ação) e Sports (esportes) são diferentes.

**Hipóteses:**

* **H0**: As classificações médias de usuários para os gêneros Action (ação) e Sports (esportes) são iguais.
* **H1**: As classificações médias de usuários para os gêneros Action (ação) e Sports (esportes) são diferentes.
* Utilizaremos alpha 0.05 por ser um padrão amplamente aceito (e as observações anteriores indicarem uma diferença expressiva nas médias)

Vamos utilizar o teste de hipótese sobre a igualdade da média de duas populações, já que estamos comparando a média de dois grupos. Temos uma diferença relativa entre o tamanho da amostragem das populações, mas variâncias próximas (verificado abaixo), por isso vamos usar o teste t de Student.

In [713]:
# Criando os subsets

df_action = df_recent[(df_recent['genre'] == 'Action') & (df_recent['user_score'].notnull())]
df_sports = df_recent[(df_recent['genre'] == 'Sports') & (df_recent['user_score'].notnull())]

display(df_action.describe())
display(df_sports.describe())

print('Variância do user_score para jogos de Action:', df_action['user_score'].var())
print('Variância do user_score para jogos de Sports:', df_sports['user_score'].var())

print()
print('Avaliação média dos usuários para jogos de Action:', df_action['user_score'].mean())
print('Avaliação média dos usuários para jogos de Sports:', df_sports['user_score'].mean())

,index,year_of_release,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,total_sales
count,779.000000,779.000000,779.000000,779.000000,779.000000,779.000000,675.000000,779.000000,779.000000
mean,6751.383825,2012.761232,0.312580,0.257112,0.043312,0.084326,69.637037,67.818999,0.697330
std,4581.660252,1.960270,0.635648,0.591106,0.112537,0.217255,13.263483,13.602013,1.413782
min,16.000000,2010.000000,0.000000,0.000000,0.000000,0.000000,24.000000,16.000000,0.010000
25%,2766.000000,2011.000000,0.040000,0.030000,0.000000,0.010000,61.000000,60.000000,0.120000
50%,6192.000000,2012.000000,0.120000,0.090000,0.000000,0.030000,72.000000,71.000000,0.280000
75%,9878.000000,2014.000000,0.330000,0.270000,0.030000,0.080000,79.000000,78.000000,0.740000
max,16692.000000,2016.000000,9.660000,9.090000,1.110000,3.960000,97.000000,91.000000,21.050000


,index,year_of_release,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,total_sales
count,314.000000,314.000000,314.000000,314.000000,314.000000,314.000000,241.000000,314.000000,314.000000
mean,5566.936306,2012.691083,0.415191,0.303185,0.019586,0.104268,72.269710,59.028662,0.842229
std,4316.494421,2.105094,0.584089,0.712282,0.069880,0.175649,14.966556,16.864688,1.175135
min,77.000000,2010.000000,0.000000,0.000000,0.000000,0.000000,19.000000,2.000000,0.010000
25%,2248.750000,2011.000000,0.070000,0.030000,0.000000,0.020000,68.000000,50.000000,0.180000
50%,4528.000000,2013.000000,0.175000,0.110000,0.000000,0.040000,77.000000,62.000000,0.430000
75%,8108.500000,2015.000000,0.505000,0.250000,0.000000,0.120000,83.000000,72.000000,0.920000
max,16643.000000,2016.000000,3.920000,6.120000,0.630000,1.370000,91.000000,90.000000,8.580000


Variância do user_score para jogos de Action: 185.014754266065
Variância do user_score para jogos de Sports: 284.41770619238514

Avaliação média dos usuários para jogos de Action: 67.81899871630296
Avaliação média dos usuários para jogos de Sports: 59.02866242038217


In [714]:
# Testando as hipóteses

# Definindo alpha
alpha = 0.05

# Criando os grupos a serem testados
score_action = df_action['user_score']
score_sports = df_sports['user_score']

# Realizando o test t de Student para comparar as médias de receita entre os dois planos
statistics, p_value = stats.ttest_ind(score_action, score_sports, equal_var=True)

# Imprimindo os resultados
print(f'Estatística: {statistics}')
print(f'P-valor: {p_value}')
print(f'Significância: {alpha}')

# Interpretando os resultados
if p_value < alpha:
    print('Rejeitamos a hipótese nula: há uma diferença estatisticamente significativa entre as médias das avaliações de Action e Sports.')
else:    
    print('Não rejeitamos a hipótese nula: não há uma diferença estatisticamente significativa entre as médias das avaliações de Action e Sports.')

Estatística: 8.999051235510347
P-valor: 9.88179232084911e-19
Significância: 0.05
Rejeitamos a hipótese nula: há uma diferença estatisticamente significativa entre as médias das avaliações de Action e Sports.


## Conclusão

A análise indica que o ecossistema PlayStation apresenta o maior potencial para campanhas em 2017. O PS4 consolidou-se como a principal plataforma em crescimento, enquanto o PS3 ainda mantém uma base relevante de usuários. Estratégias que combinem foco no PS4 com suporte aos usuários do PS3 podem ampliar o alcance dos lançamentos. Ao mesmo tempo, a forte concorrência entre plataformas reforça a importância de uma abordagem multiplataforma para atender diferentes mercados.

Os gêneros Shooter, Sports e Platform concentram as maiores oportunidades de vendas na América do Norte, Europa e demais regiões, enquanto o mercado japonês apresenta preferências distintas, com destaque para RPG e Adventure. Essas diferenças regionais sugerem a necessidade de campanhas segmentadas por público e localização.

A classificação ESRB também se mostrou um fator relevante para o planejamento. Jogos classificados como M (Mature) apresentaram melhor desempenho na América do Norte e Europa, enquanto títulos E (Everyone) tiveram participação relativamente maior no Japão. Dessa forma, alinhar a comunicação e o posicionamento dos jogos às preferências de cada mercado pode maximizar o retorno das campanhas.

As avaliações demonstraram influência moderada nas vendas, com destaque para as avaliações de críticos, que apresentaram correlação positiva mais consistente. Isso sugere que ações voltadas para aumentar a exposição dos jogos à crítica especializada podem contribuir para o desempenho comercial dos lançamentos.

Os testes de hipótese indicaram que as avaliações dos usuários para as plataformas Xbox One e PC não apresentam diferenças estatisticamente significativas, sugerindo que campanhas direcionadas a esses públicos podem adotar estratégias semelhantes de comunicação e posicionamento. Por outro lado, foram observadas diferenças significativas entre as avaliações dos gêneros Action e Sports, evidenciando que a recepção dos usuários varia conforme o tipo de experiência oferecida. Esse resultado reforça a necessidade de adaptar a mensagem de marketing às características de cada gênero, mesmo quando os títulos disputam o mesmo mercado.